In [ ]:
# W9.2 Gate A — Existence + discover prediction-like CSVs under results/
from __future__ import annotations

from pathlib import Path
import os
import csv

def find_project_root(start: Path) -> Path:

    cur = start.resolve()
    for _ in range(10):  
        if (cur / "config" / "labels_features.yaml").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise FileNotFoundError("Could not locate project root ")

def sizeof(path: Path) -> int:
    try:
        return path.stat().st_size
    except Exception:
        return -1

def sniff_csv_header(path: Path, max_bytes: int = 2_000_000) -> list[str] | None:

    try:
        if sizeof(path) > max_bytes:
            pass
        with path.open("r", encoding="utf-8", newline="") as f:
            reader = csv.reader(f)
            header = next(reader, None)
            if header is None:
                return None
            return [h.strip() for h in header]
    except UnicodeDecodeError:
        try:
            with path.open("r", encoding="latin-1", newline="") as f:
                reader = csv.reader(f)
                header = next(reader, None)
                if header is None:
                    return None
                return [h.strip() for h in header]
        except Exception:
            return None
    except Exception:
        return None

try:
    start_dir = Path.cwd()
    ROOT = find_project_root(start_dir)
except Exception as e:
    print("Not provable from provided evidence.")
    print(f"Root inference failed from Path.cwd()={Path.cwd()!s}")
    print(f"Error: {type(e).__name__}: {e}")
    raise

print(f"[Gate A] Path.cwd(): {Path.cwd()}")
print(f"[Gate A] Project ROOT: {ROOT}")

# 2) Check required paths
required_rel = [
    Path("config/labels_features.yaml"),
    Path("data/processed/mru_usd_ohlc_clean.csv"),
    Path("docs/week07/week07_backtesting_design.md"),
    Path("docs/week04_methodology/validation_splits.md"),
    Path("docs/week04_methodology/data_cleaning_spec.md"),
    Path("results"),
]

print("\n[Gate A] Required path existence:")
for rel in required_rel:
    p = ROOT / rel
    print(f"- {rel.as_posix():55s} exists={p.exists()}  type={'dir' if p.is_dir() else 'file' if p.is_file() else 'missing'}")

results_dir = ROOT / "results"
csv_files: list[Path] = []
if results_dir.exists() and results_dir.is_dir():
    csv_files = sorted([p for p in results_dir.rglob("*.csv") if p.is_file()])
else:
    print("\nNot provable from provided evidence.")
    raise FileNotFoundError("results/ missing")

print(f"\n[Gate A] CSV files under results/ (count={len(csv_files)}):")
for p in csv_files:
    rel = p.relative_to(ROOT).as_posix()
    print(f"- {rel}  size_bytes={sizeof(p)}")

pred_key_cols = {"y_pred", "pred", "prediction"}
date_col_names = {"date", "Date", "DATE", "timestamp", "datetime"} 

candidates = []
for p in csv_files:
    header = sniff_csv_header(p)
    if not header:
        continue
    header_lc = [h.lower() for h in header]
    has_date = ("date" in header_lc) or any(h in date_col_names for h in header)
    has_predish = any(k in header_lc for k in pred_key_cols) or any(k in header for k in pred_key_cols)
    if has_date and has_predish:
        candidates.append((p, header))

print(f"\n[Gate A] Prediction-like CSV candidates (date + one of y_pred/pred/prediction) count={len(candidates)}:")
if candidates:
    for p, header in candidates:
        rel = p.relative_to(ROOT).as_posix()
        cols_preview = ", ".join(header[:30]) + (" ..." if len(header) > 30 else "")
        print(f"- {rel}\n  columns: {cols_preview}")
else:
    print("None found by header sniffing rule.")



In [ ]:
# W9.2 Gate B — Schema + integrity checks for prediction CSVs 
from __future__ import annotations

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path(r"C:\Users\simon\Desktop\my thesis\ThesisProject")
pred_dir = ROOT / "results" / "predictions"

if not pred_dir.exists():
    print("Not provable from provided evidence.")
    print(f"Missing folder: {pred_dir}")
    raise FileNotFoundError(pred_dir)

pred_files = sorted(pred_dir.glob("*.csv"))
if not pred_files:
    print("Not provable from provided evidence.")
    print(f"No CSV files found in: {pred_dir}")
    raise FileNotFoundError("No prediction CSVs")

REQUIRED_COLS = ["date", "y_true", "y_pred"]
ALLOWED_LABELS = {-1, 0, 1}

rows = []
failures = []

for p in pred_files:
    rel = p.relative_to(ROOT).as_posix()
    try:
        df = pd.read_csv(p)
    except Exception as e:
        failures.append((rel, f"read_csv failed: {type(e).__name__}: {e}"))
        continue

    cols = list(df.columns)
    missing = [c for c in REQUIRED_COLS if c not in cols]
    if missing:
        failures.append((rel, f"missing required cols: {missing} | cols={cols}"))
        continue

    # date parse
    date = pd.to_datetime(df["date"], errors="coerce")
    date_nat = int(date.isna().sum())

    # y_true / y_pred checks (numeric + allowed set)
    y_true = pd.to_numeric(df["y_true"], errors="coerce")
    y_pred = pd.to_numeric(df["y_pred"], errors="coerce")

    y_true_nan = int(y_true.isna().sum())
    y_pred_nan = int(y_pred.isna().sum())

    # Convert to ints where possible for set checks
    y_true_vals = set(pd.Series(y_true.dropna().unique()).astype(int).tolist())
    y_pred_vals = set(pd.Series(y_pred.dropna().unique()).astype(int).tolist())

    y_true_ok = y_true_vals.issubset(ALLOWED_LABELS)
    y_pred_ok = y_pred_vals.issubset(ALLOWED_LABELS)

    # date integrity
    dup_dates = int(pd.Series(date).duplicated().sum())
    mono_inc = bool(pd.Series(date).is_monotonic_increasing)

    # basic stats
    n = int(len(df))
    date_min = str(pd.Series(date).min()) if n else "NA"
    date_max = str(pd.Series(date).max()) if n else "NA"

    rows.append({
        "file": rel,
        "n_rows": n,
        "cols": ",".join(cols),
        "date_NaT": date_nat,
        "dup_dates": dup_dates,
        "mono_increasing": mono_inc,
        "y_true_nan": y_true_nan,
        "y_pred_nan": y_pred_nan,
        "y_true_vals": sorted(list(y_true_vals)),
        "y_pred_vals": sorted(list(y_pred_vals)),
        "y_true_ok": y_true_ok,
        "y_pred_ok": y_pred_ok,
        "date_min": date_min,
        "date_max": date_max,
    })

    if date_nat > 0:
        failures.append((rel, f"date parse produced NaT count={date_nat}"))
    if dup_dates > 0:
        failures.append((rel, f"duplicate dates detected count={dup_dates}"))
    if (y_true_nan > 0) or (y_pred_nan > 0):
        failures.append((rel, f"NaNs detected: y_true_nan={y_true_nan}, y_pred_nan={y_pred_nan}"))
    if not y_true_ok:
        failures.append((rel, f"y_true values outside {-1,0,1}: {sorted(list(y_true_vals))}"))
    if not y_pred_ok:
        failures.append((rel, f"y_pred values outside {-1,0,1}: {sorted(list(y_pred_vals))}"))

summary = pd.DataFrame(rows).sort_values(["file"]).reset_index(drop=True)

print(f"[Gate B] Prediction CSV count: {len(pred_files)}")
with pd.option_context("display.max_rows", 300, "display.max_colwidth", 120, "display.width", 180):
    print(summary[[
        "file","n_rows","date_min","date_max","date_NaT","dup_dates","mono_increasing",
        "y_true_nan","y_pred_nan","y_true_vals","y_pred_vals","y_true_ok","y_pred_ok"
    ]])

if failures:
    print("\n[Gate B] FAILURES (STOP):")
    for f, msg in failures:
        print(f"- {f}: {msg}")
    raise RuntimeError("Gate B failed ")
else:
    print("\n[Gate B] PASS: all prediction CSVs have required schema and basic integrity checks.")



In [ ]:
# W9.2 Gate C — Price alignment checks (predictions ↔ OHLC)
from __future__ import annotations

from pathlib import Path
import pandas as pd

CWD = Path.cwd().resolve()
ROOT = CWD
while ROOT.name.lower() != "thesisproject" and ROOT.parent != ROOT:
    ROOT = ROOT.parent

pred_dir = ROOT / "results" / "predictions"
ohlc_path = ROOT / "data" / "processed" / "mru_usd_ohlc_clean.csv"

print(f"[Gate C] Path.cwd(): {CWD}")
print(f"[Gate C] Project ROOT: {ROOT}")

if not pred_dir.exists():
    print("Not provable from provided evidence.")
    print(f"Missing folder: {pred_dir}")
    raise FileNotFoundError(pred_dir)

if not ohlc_path.exists():
    print("Not provable from provided evidence.")
    print(f"Missing file: {ohlc_path}")
    raise FileNotFoundError(ohlc_path)

# Load OHLC (minimal columns, but we need schema proof)
ohlc = pd.read_csv(ohlc_path)
print("\n[Gate C] OHLC columns:", list(ohlc.columns))

required_ohlc_cols = ["date", "close"]
missing = [c for c in required_ohlc_cols if c not in ohlc.columns]
if missing:
    print("Not provable from provided evidence.")
    print(f"OHLC missing required columns: {missing}")
    raise ValueError("OHLC schema mismatch")

ohlc["date"] = pd.to_datetime(ohlc["date"], errors="coerce")
nat = int(ohlc["date"].isna().sum())
dup = int(ohlc["date"].duplicated().sum())

print(f"[Gate C] OHLC rows: {len(ohlc)} | date_NaT: {nat} | dup_dates: {dup}")
print(f"[Gate C] OHLC date range: {ohlc['date'].min()} → {ohlc['date'].max()}")

if nat > 0 or dup > 0:
    raise RuntimeError("Gate C failed — fix OHLC date column integrity first.")

# Build a set of tradable dates (OHLC date index)
ohlc_dates = set(ohlc["date"].dt.normalize())

# Align each prediction file to OHLC dates
pred_files = sorted(pred_dir.glob("*.csv"))
rows = []
failures = []

for p in pred_files:
    rel = p.relative_to(ROOT).as_posix()
    df = pd.read_csv(p, usecols=["date", "y_true", "y_pred"])
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    if int(df["date"].isna().sum()) > 0:
        failures.append((rel, "prediction file has NaT dates"))
        continue
    if int(df["date"].duplicated().sum()) > 0:
        failures.append((rel, "prediction file has duplicate dates"))
        continue

    pred_dates = df["date"].dt.normalize()
    missing_in_ohlc = sorted(list(set(pred_dates) - ohlc_dates))
    n_missing = len(missing_in_ohlc)

    rows.append({
        "file": rel,
        "pred_rows": int(len(df)),
        "pred_date_min": str(pred_dates.min()),
        "pred_date_max": str(pred_dates.max()),
        "missing_pred_dates_in_ohlc": n_missing,
    })

    if n_missing > 0:
        failures.append((rel, f"pred dates missing in OHLC (count={n_missing}) first5={missing_in_ohlc[:5]}"))

summary = pd.DataFrame(rows).sort_values("file").reset_index(drop=True)

print("\n[Gate C] Prediction ↔ OHLC date coverage summary:")
with pd.option_context("display.max_rows", 300, "display.width", 180):
    print(summary)

print("\n[Gate C] Last 10 OHLC dates:")
print(pd.Series(sorted(list(ohlc_dates))[-10:]).to_string(index=False))

if failures:
    print("\n[Gate C] FAILURES (STOP):")
    for f, msg in failures:
        print(f"- {f}: {msg}")
    raise RuntimeError("Gate C failed ")
else:
    print("\n[Gate C] PASS: all prediction dates exist in OHLC ")



In [ ]:
# Gate D — Window containment 
from pathlib import Path
import re
import pandas as pd

cwd = Path.cwd()
ROOT = cwd
while ROOT.name != "ThesisProject" and ROOT.parent != ROOT:
    ROOT = ROOT.parent

print(f"[Gate D] Path.cwd(): {cwd}")
print(f"[Gate D] Project ROOT: {ROOT}")

pred_dir = ROOT / "results" / "predictions"
ohlc_path = ROOT / "data" / "processed" / "mru_usd_ohlc_clean.csv"

ohlc = pd.read_csv(ohlc_path, parse_dates=["date"])
ohlc_min, ohlc_max = ohlc["date"].min(), ohlc["date"].max()
print(f"[Gate D] OHLC date range: {ohlc_min} → {ohlc_max}")

WINDOWS = {
    "V1_test": (pd.Timestamp("2020-01-01"), pd.Timestamp("2025-11-25")),
    "V2_F1_test": (pd.Timestamp("2010-01-01"), pd.Timestamp("2011-12-31")),
    "V2_F2_test": (pd.Timestamp("2012-01-01"), pd.Timestamp("2013-12-31")),
    "V2_F3_test": (pd.Timestamp("2014-01-01"), pd.Timestamp("2015-12-31")),
    "V2_F4_test": (pd.Timestamp("2016-01-01"), pd.Timestamp("2019-12-31")),
}

def infer_declared_window_key(fname: str) -> str:

    if "_V1_test" in fname:
        return "V1_test"
    m = re.search(r"_V2_(F[1-4])_test", fname)
    if m:
        return f"V2_{m.group(1)}_test"
    return "UNKNOWN"

pred_files = sorted(pred_dir.glob("*.csv"))
if not pred_files:
    raise FileNotFoundError(f"No CSVs found under: {pred_dir}")

rows = []
clip_actions = []

for p in pred_files:
    df = pd.read_csv(p, usecols=["date"])  
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    file_min = df["date"].min()
    file_max = df["date"].max()
    n_rows = len(df)

    key = infer_declared_window_key(p.name)
    if key in WINDOWS:
        w_start, w_end = WINDOWS[key]
        ok = (file_min >= w_start) and (file_max <= w_end)
        declared = f"{key}:{w_start.date()}→{w_end.date()}"
        clipped_n = 0

        if not ok:
            full = pd.read_csv(p, parse_dates=["date"])
            before = len(full)
            clipped = full[(full["date"] >= w_start) & (full["date"] <= w_end)].copy()
            after = len(clipped)
            clipped_n = before - after

            out_path = p.with_name(p.stem + "__clipped.csv")
            clipped.to_csv(out_path, index=False)
            clip_actions.append((str(p.relative_to(ROOT)), str(out_path.relative_to(ROOT)), before, after, clipped_n))

            file_min = clipped["date"].min() if after > 0 else file_min
            file_max = clipped["date"].max() if after > 0 else file_max
            ok = (file_min >= w_start) and (file_max <= w_end)
    else:
        declared = "UNKNOWN"
        ok = False
        clipped_n = None

    rows.append({
        "file": str(p.relative_to(ROOT)),
        "declared_window": declared,
        "file_min": file_min,
        "file_max": file_max,
        "n_rows": n_rows,
        "OK": bool(ok),
    })

containment = pd.DataFrame(rows).sort_values(["declared_window", "file"]).reset_index(drop=True)
print("\n[Gate D] Containment table:")
print(containment.to_string(index=False))

if clip_actions:
    print("\n[Gate D] CLIPPING ACTIONS ")
    for orig, outp, before, after, dropped in clip_actions:
        print(f"- {orig} → {outp} | rows {before} → {after} | clipped_out={dropped}")
else:
    print("\n[Gate D] No clipping required (")




In [ ]:
# W9.3 — Canonical return series (simple returns) + window coverage checks
from pathlib import Path
import pandas as pd

cwd = Path.cwd()
ROOT = cwd
while ROOT.name != "ThesisProject" and ROOT.parent != ROOT:
    ROOT = ROOT.parent

print(f"[W9.3] Path.cwd(): {cwd}")
print(f"[W9.3] Project ROOT: {ROOT}")

ohlc_path = ROOT / "data" / "processed" / "mru_usd_ohlc_clean.csv"
pred_dir = ROOT / "results" / "predictions"

ohlc = pd.read_csv(ohlc_path, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
assert list(ohlc.columns) == ["date", "open", "high", "low", "close"], f"Unexpected OHLC columns: {ohlc.columns.tolist()}"

ohlc["ret_simple"] = ohlc["close"].pct_change()

print(f"[W9.3] OHLC rows: {len(ohlc)} | date range: {ohlc['date'].min()} → {ohlc['date'].max()}")
print(f"[W9.3] ret_simple NaNs: {ohlc['ret_simple'].isna().sum()} (expected 1 at first row)")

WINDOWS = {
    "V1_test": (pd.Timestamp("2020-01-01"), pd.Timestamp("2025-11-25")),
    "V2_F1_test": (pd.Timestamp("2010-01-01"), pd.Timestamp("2011-12-31")),
    "V2_F2_test": (pd.Timestamp("2012-01-01"), pd.Timestamp("2013-12-31")),
    "V2_F3_test": (pd.Timestamp("2014-01-01"), pd.Timestamp("2015-12-31")),
    "V2_F4_test": (pd.Timestamp("2016-01-01"), pd.Timestamp("2019-12-31")),
}

def in_window(df, start, end):
    return df[(df["date"] >= start) & (df["date"] <= end)].copy()

rows = []
for name, (start, end) in WINDOWS.items():
    w = in_window(ohlc, start, end)
    rows.append({
        "window": name,
        "start": start,
        "end": end,
        "ohlc_rows": len(w),
        "ret_nan_in_window": int(w["ret_simple"].isna().sum()),
        "ret_min_date": w.loc[w["ret_simple"].notna(), "date"].min() if len(w) else None,
        "ret_max_date": w.loc[w["ret_simple"].notna(), "date"].max() if len(w) else None,
    })

cov = pd.DataFrame(rows).sort_values("window").reset_index(drop=True)
print("\n[W9.3] Return coverage by window:")
print(cov.to_string(index=False))

prev_check = []
for name, (start, end) in WINDOWS.items():
    has_start = (ohlc["date"] == start).any()
    if has_start:
        idx = int(ohlc.index[ohlc["date"] == start][0])
        prev_exists = (idx - 1) >= 0
        ret_at_start = ohlc.loc[idx, "ret_simple"]
        prev_check.append({
            "window": name,
            "start": start,
            "start_in_ohlc": True,
            "prev_row_exists": bool(prev_exists),
            "ret_simple_at_start_is_nan": bool(pd.isna(ret_at_start)),
        })
    else:
        prev_check.append({
            "window": name,
            "start": start,
            "start_in_ohlc": False,
            "prev_row_exists": False,
            "ret_simple_at_start_is_nan": None,
        })

prev_df = pd.DataFrame(prev_check).sort_values("window").reset_index(drop=True)
print("\n[W9.3] Start-of-window return sanity (ret_simple at start should NOT be NaN):")
print(prev_df.to_string(index=False))

# --- Print minimal head/tail for audit ---
print("\n[W9.3] OHLC+ret_simple head (first 5):")
print(ohlc[["date", "close", "ret_simple"]].head(5).to_string(index=False))

print("\n[W9.3] OHLC+ret_simple tail (last 5):")
print(ohlc[["date", "close", "ret_simple"]].tail(5).to_string(index=False))




In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

cwd = Path.cwd()
ROOT = cwd
while ROOT.name != "ThesisProject" and ROOT.parent != ROOT:
    ROOT = ROOT.parent

print(f"[W9.4] Path.cwd(): {cwd}")
print(f"[W9.4] Project ROOT: {ROOT}")

ohlc_path = ROOT / "data" / "processed" / "mru_usd_ohlc_clean.csv"
pred_path = ROOT / "results" / "predictions" / "predictions_E1_V1_test.csv"

V1_START = pd.Timestamp("2020-01-01")
V1_END   = pd.Timestamp("2025-11-25")

ohlc = pd.read_csv(ohlc_path, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
ohlc["ret_simple"] = ohlc["close"].pct_change()

ohlc_w = ohlc[(ohlc["date"] >= V1_START) & (ohlc["date"] <= V1_END)].copy()
assert len(ohlc_w) > 0, "Empty OHLC window"
assert ohlc_w["ret_simple"].isna().sum() == 0, "Unexpected NaN returns inside V1_test window"

pred = pd.read_csv(pred_path, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
expected_cols = {"date", "y_true", "y_pred"}
assert set(pred.columns) == expected_cols, f"Unexpected prediction columns: {pred.columns.tolist()}"

df = ohlc_w.merge(pred, on="date", how="left", validate="one_to_one")

df["signal_raw"] = df["y_pred"].fillna(0).astype(int)

df["position"] = df["signal_raw"].shift(1).fillna(0).astype(int)

df["strategy_ret"] = df["position"] * df["ret_simple"]

df["equity"] = (1.0 + df["strategy_ret"]).cumprod()

print(f"\n[W9.4] Run: T1_L1Q_V1 (E1)")
print(f"[W9.4] Declared window: {V1_START.date()} → {V1_END.date()}")
print(f"[W9.4] OHLC window rows: {len(ohlc_w)} | OHLC min/max: {ohlc_w['date'].min().date()} → {ohlc_w['date'].max().date()}")
print(f"[W9.4] Prediction file rows: {len(pred)} | pred min/max: {pred['date'].min().date()} → {pred['date'].max().date()}")

n_missing_pred = int(df["y_pred"].isna().sum())
print(f"[W9.4] Missing y_pred after alignment (filled as 0 for signal): {n_missing_pred} / {len(df)}")

print("\n[W9.4] Distribution checks:")
print("signal_raw value counts:\n", df["signal_raw"].value_counts(dropna=False).sort_index())
print("position value counts:\n", df["position"].value_counts(dropna=False).sort_index())
exposure = float((df["position"] != 0).mean())
print(f"[W9.4] Exposure (position != 0): {exposure:.6f}")

cum_return = float(df["equity"].iloc[-1] - 1.0)
mu = float(df["strategy_ret"].mean())
sigma = float(df["strategy_ret"].std(ddof=1))
ann_vol = sigma * np.sqrt(252.0) if sigma > 0 else np.nan
sharpe = (mu / sigma) * np.sqrt(252.0) if sigma > 0 else np.nan

running_max = df["equity"].cummax()
drawdown = df["equity"] / running_max - 1.0
max_dd = float(drawdown.min())

pos_change = (df["position"] != df["position"].shift(1)).fillna(False)
n_pos_changes = int(pos_change.sum())

n_entries = int(((df["position"] != 0) & (df["position"].shift(1).fillna(0) == 0)).sum())
n_exits   = int(((df["position"] == 0) & (df["position"].shift(1).fillna(0) != 0)).sum())
n_flips   = int(((df["position"] != 0) & (df["position"].shift(1).fillna(0) != 0) &
                 (np.sign(df["position"]) != np.sign(df["position"].shift(1).fillna(0)))).sum())

print("\n[W9.4] Core metrics (win rate NOT computed yet):")
print(pd.DataFrame([{
    "cum_return": cum_return,
    "ann_vol": ann_vol,
    "sharpe_rf0": sharpe,
    "max_drawdown": max_dd,
    "n_position_changes": n_pos_changes,
    "n_entries": n_entries,
    "n_exits": n_exits,
    "n_flips": n_flips,
}]).to_string(index=False))

cols = ["date", "close", "ret_simple", "y_pred", "signal_raw", "position", "strategy_ret", "equity"]
print("\n[W9.4] Sanity table HEAD (first 8 rows):")
print(df[cols].head(8).to_string(index=False))

print("\n[W9.4] Sanity table TAIL (last 8 rows):")
print(df[cols].tail(8).to_string(index=False))




In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("..").resolve()
OHLC_PATH = ROOT / "data" / "processed" / "mru_usd_ohlc_clean.csv"

print(f"[W9.3_LOG] Project ROOT: {ROOT}")

WINDOWS = {
    "V1_test": ("2020-01-01", "2025-11-25"),
    "V2_F1_test": ("2010-01-01", "2011-12-31"),
    "V2_F2_test": ("2012-01-01", "2013-12-31"),
    "V2_F3_test": ("2014-01-01", "2015-12-31"),
    "V2_F4_test": ("2016-01-01", "2019-12-31"),
}

ohlc = pd.read_csv(OHLC_PATH, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
ohlc["log_close"] = np.log(ohlc["close"].astype(float))
ohlc["ret_log"] = ohlc["log_close"].diff()  
ohlc["ret_simple_derived"] = np.exp(ohlc["ret_log"]) - 1  

print(f"[W9.3_LOG] OHLC rows: {len(ohlc)} | date range: {ohlc['date'].min()} → {ohlc['date'].max()}")
print(f"[W9.3_LOG] ret_log NaNs: {ohlc['ret_log'].isna().sum()} (expected 1 at first row)")

rows = []
for name, (s, e) in WINDOWS.items():
    s = pd.Timestamp(s); e = pd.Timestamp(e)
    win = ohlc[(ohlc["date"] >= s) & (ohlc["date"] <= e)].copy()
    rows.append({
        "window": name,
        "start": s.date(),
        "end": e.date(),
        "ohlc_rows": len(win),
        "ret_log_nan_in_window": int(win["ret_log"].isna().sum()),
        "ret_min_date": (win["date"].min().date() if len(win) else None),
        "ret_max_date": (win["date"].max().date() if len(win) else None),
    })

cov = pd.DataFrame(rows)
print("\n[W9.3_LOG] Return coverage by window :")
print(cov.to_string(index=False))

rows2 = []
for name, (s, _) in WINDOWS.items():
    s = pd.Timestamp(s)
    idx = ohlc.index[ohlc["date"] == s].tolist()
    if not idx:
        rows2.append({"window": name, "start": s.date(), "start_in_ohlc": False, "prev_row_exists": False, "ret_log_at_start_is_nan": None})
        continue
    i = idx[0]
    prev_ok = i > 0
    rows2.append({
        "window": name,
        "start": s.date(),
        "start_in_ohlc": True,
        "prev_row_exists": prev_ok,
        "ret_log_at_start_is_nan": (bool(pd.isna(ohlc.loc[i, "ret_log"])) if prev_ok else None),
    })
san = pd.DataFrame(rows2)
print(san.to_string(index=False))


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("..").resolve()
OHLC_PATH = ROOT / "data" / "processed" / "mru_usd_ohlc_clean.csv"
PRED_PATH = ROOT / "results" / "predictions" / "predictions_E1_V1_test.csv"

V1_START = pd.Timestamp("2020-01-01")
V1_END   = pd.Timestamp("2025-11-25")

def compute_roundtrip_trades(df: pd.DataFrame):

    pos = df["position"].to_numpy()
    strat = df["strat_ret_log"].to_numpy()

    trades = []
    i = 0
    n = len(df)
    while i < n:
        if pos[i] == 0:
            i += 1
            continue
        direction = pos[i]
        j = i
        while j + 1 < n and pos[j + 1] == direction:
            j += 1
        pnl = float(np.nansum(strat[i:j+1]))
        trades.append({"t_in": df.loc[i, "date"], "t_out_boundary": (df.loc[j+1, "date"] if j+1 < n else df.loc[j, "date"]), "dir": int(direction), "pnl_log": pnl})
        i = j + 1
    trades_df = pd.DataFrame(trades)
    return trades_df

def backtest_T1_log_core(pred_csv: Path, window_start: pd.Timestamp, window_end: pd.Timestamp):
    ohlc = pd.read_csv(OHLC_PATH, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
    ohlc["log_close"] = np.log(ohlc["close"].astype(float))
    ohlc["ret_log"] = ohlc["log_close"].diff()

    win = ohlc[(ohlc["date"] >= window_start) & (ohlc["date"] <= window_end)].copy().reset_index(drop=True)

    pred = pd.read_csv(pred_csv, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
    if "y_pred" not in pred.columns:
        raise ValueError(f"Missing y_pred column in {pred_csv}")

    merged = win.merge(pred[["date", "y_pred"]], on="date", how="left")
    missing_pred = int(merged["y_pred"].isna().sum())

    merged["signal_t"] = merged["y_pred"].fillna(0).astype(int)
    merged["position"] = merged["signal_t"].shift(1).fillna(0).astype(int)
    merged.loc[0, "position"] = 0  # hard rule: flat on first day of window

    merged["strat_ret_log"] = merged["position"].astype(float) * merged["ret_log"].astype(float)
    merged["equity"] = np.exp(merged["strat_ret_log"].fillna(0.0).cumsum())

    exposure_rate = float((merged["position"] != 0).mean())
    turnover = float(merged["position"].diff().abs().fillna(0.0).mean())

    pos_diff = merged["position"].diff()
    n_pos_change_events = int((pos_diff.iloc[1:] != 0).sum())
    n_flips = int((pos_diff.abs().iloc[1:] == 2).sum())

    trades = compute_roundtrip_trades(merged)
    n_roundtrips = int(len(trades))
    if n_roundtrips == 0:
        win_rate = np.nan
        n_wins = 0; n_losses = 0; n_zero = 0
    else:
        n_wins = int((trades["pnl_log"] > 0).sum())
        n_losses = int((trades["pnl_log"] < 0).sum())
        n_zero = int((trades["pnl_log"] == 0).sum())
        win_rate = n_wins / n_roundtrips

    cum_log = float(np.nansum(merged["strat_ret_log"]))
    cum_simple = float(np.exp(cum_log) - 1.0)

    daily_std = float(np.nanstd(merged["strat_ret_log"].to_numpy(), ddof=0))
    daily_mean = float(np.nanmean(merged["strat_ret_log"].to_numpy()))
    ann_vol = daily_std * float(np.sqrt(252.0))
    sharpe = (daily_mean / daily_std * float(np.sqrt(252.0))) if daily_std > 0 else np.nan

    eq = merged["equity"].astype(float)
    dd = (eq / eq.cummax()) - 1.0
    max_dd = float(dd.min())

    summary = {
        "cumulative_log_return": cum_log,
        "cumulative_simple_return": cum_simple,
        "annualized_vol_ddof0": ann_vol,
        "sharpe_like_rf0_ddof0": sharpe,
        "max_drawdown": max_dd,
        "exposure_rate": exposure_rate,
        "turnover": turnover,
        "n_roundtrip_trades": n_roundtrips,
        "n_position_change_events": n_pos_change_events,
        "n_flips": n_flips,
        "win_rate": win_rate,
        "n_wins": n_wins,
        "n_losses": n_losses,
        "n_zero_pnl_trades": n_zero,
        "missing_y_pred_filled_as_0": missing_pred,
    }
    return merged, trades, summary

print(f"[W9.4_LOG] Run: T1_L1Q_V1 (E1)")
print(f"[W9.4_LOG] Declared window: {V1_START.date()} → {V1_END.date()}")

bt_df, trades_df, summary = backtest_T1_log_core(PRED_PATH, V1_START, V1_END)

print(f"[W9.4_LOG] OHLC window rows: {len(bt_df)} | OHLC min/max: {bt_df['date'].min().date()} → {bt_df['date'].max().date()}")
print(f"[W9.4_LOG] Prediction file rows: {pd.read_csv(PRED_PATH).shape[0]}")
print(f"[W9.4_LOG] Missing y_pred after alignment (filled as 0 for signal): {summary['missing_y_pred_filled_as_0']} / {len(bt_df)}")

print("\n[W9.4_LOG] Distribution checks:")
print("signal_t value counts:")
print(bt_df["signal_t"].value_counts(dropna=False).sort_index())
print("position value counts:")
print(bt_df["position"].value_counts(dropna=False).sort_index())
print(f"[W9.4_LOG] Exposure (position != 0): {summary['exposure_rate']:.6f}")

print("\n[W9.4_LOG] Core metrics (log-core; cost=0):")
for k in [
    "cumulative_log_return","cumulative_simple_return","annualized_vol_ddof0","sharpe_like_rf0_ddof0",
    "max_drawdown","exposure_rate","turnover",
    "n_roundtrip_trades","n_position_change_events","n_flips","win_rate","n_wins","n_losses","n_zero_pnl_trades"
]:
    print(f"{k}: {summary[k]}")

print("\n[W9.4_LOG] Sanity table HEAD (first 8 rows):")
cols = ["date","close","ret_log","y_pred","signal_t","position","strat_ret_log","equity"]
print(bt_df[cols].head(8).to_string(index=False))

print("\n[W9.4_LOG] Sanity table TAIL (last 8 rows):")
print(bt_df[cols].tail(8).to_string(index=False))

print("\n[W9.4_LOG] Trades head (if any):")
print(trades_df.head(10).to_string(index=False) if len(trades_df) else "NO TRADES")


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("..").resolve()
OHLC_PATH = ROOT / "data" / "processed" / "mru_usd_ohlc_clean.csv"
PRED_PATH = ROOT / "results" / "predictions" / "predictions_E5_V1_test.csv"

V1_START = pd.Timestamp("2020-01-01")
V1_END   = pd.Timestamp("2025-11-25")

def compute_roundtrip_trades(df: pd.DataFrame):
    pos = df["position"].to_numpy()
    strat = df["strat_ret_log"].to_numpy()

    trades = []
    i = 0
    n = len(df)
    while i < n:
        if pos[i] == 0:
            i += 1
            continue
        direction = pos[i]
        j = i
        while j + 1 < n and pos[j + 1] == direction:
            j += 1
        pnl = float(np.nansum(strat[i:j+1]))
        trades.append({
            "t_in": df.loc[i, "date"],
            "t_out_boundary": (df.loc[j+1, "date"] if j+1 < n else df.loc[j, "date"]),
            "dir": int(direction),
            "pnl_log": pnl
        })
        i = j + 1
    return pd.DataFrame(trades)

def backtest_log_core(pred_csv: Path, window_start: pd.Timestamp, window_end: pd.Timestamp):
    ohlc = pd.read_csv(OHLC_PATH, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
    ohlc["log_close"] = np.log(ohlc["close"].astype(float))
    ohlc["ret_log"] = ohlc["log_close"].diff()

    win = ohlc[(ohlc["date"] >= window_start) & (ohlc["date"] <= window_end)].copy().reset_index(drop=True)

    pred = pd.read_csv(pred_csv, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
    if "y_pred" not in pred.columns:
        raise ValueError(f"Missing y_pred column in {pred_csv}")

    merged = win.merge(pred[["date", "y_pred"]], on="date", how="left")
    missing_pred = int(merged["y_pred"].isna().sum())

    # signal at t is y_pred(t); tradable position at t uses y_pred(t-1)
    merged["signal_t"] = merged["y_pred"].fillna(0).astype(int)

    # HARD RULE: first day flat
    merged["position"] = merged["signal_t"].shift(1).fillna(0).astype(int)
    merged.loc[0, "position"] = 0

    # LOG-core returns, cost=0
    merged["strat_ret_log"] = merged["position"].astype(float) * merged["ret_log"].astype(float)
    merged["equity"] = np.exp(merged["strat_ret_log"].fillna(0.0).cumsum())

    # exposure + turnover
    exposure_rate = float((merged["position"] != 0).mean())
    turnover = float(merged["position"].diff().abs().fillna(0.0).mean())

    # position-change events
    pos_diff = merged["position"].diff()
    n_pos_change_events = int((pos_diff.iloc[1:] != 0).sum())
    n_flips = int((pos_diff.abs().iloc[1:] == 2).sum())

    # round-trip trades + win rate
    trades = compute_roundtrip_trades(merged)
    n_roundtrips = int(len(trades))
    if n_roundtrips == 0:
        win_rate = np.nan
        n_wins = 0; n_losses = 0; n_zero = 0
    else:
        n_wins = int((trades["pnl_log"] > 0).sum())
        n_losses = int((trades["pnl_log"] < 0).sum())
        n_zero = int((trades["pnl_log"] == 0).sum())
        win_rate = n_wins / n_roundtrips

    cum_log = float(np.nansum(merged["strat_ret_log"]))
    cum_simple = float(np.exp(cum_log) - 1.0)

    # deterministic ddof=0
    x = merged["strat_ret_log"].to_numpy()
    daily_std = float(np.nanstd(x, ddof=0))
    daily_mean = float(np.nanmean(x))
    ann_vol = daily_std * float(np.sqrt(252.0))
    sharpe = (daily_mean / daily_std * float(np.sqrt(252.0))) if daily_std > 0 else np.nan

    # max drawdown
    eq = merged["equity"].astype(float)
    dd = (eq / eq.cummax()) - 1.0
    max_dd = float(dd.min())

    summary = {
        "cumulative_log_return": cum_log,
        "cumulative_simple_return": cum_simple,
        "annualized_vol_ddof0": ann_vol,
        "sharpe_like_rf0_ddof0": sharpe,
        "max_drawdown": max_dd,
        "exposure_rate": exposure_rate,
        "turnover": turnover,
        "n_roundtrip_trades": n_roundtrips,
        "n_position_change_events": n_pos_change_events,
        "n_flips": n_flips,
        "win_rate": win_rate,
        "n_wins": n_wins,
        "n_losses": n_losses,
        "n_zero_pnl_trades": n_zero,
        "missing_y_pred_filled_as_0": missing_pred,
    }
    return merged, trades, summary

print(f"[W9.4_LOG] Run: T2_L2_V1 (E5)")
print(f"[W9.4_LOG] Declared window: {V1_START.date()} → {V1_END.date()}")

bt_df, trades_df, summary = backtest_log_core(PRED_PATH, V1_START, V1_END)

print(f"[W9.4_LOG] OHLC window rows: {len(bt_df)} | OHLC min/max: {bt_df['date'].min().date()} → {bt_df['date'].max().date()}")
print(f"[W9.4_LOG] Prediction file rows: {pd.read_csv(PRED_PATH).shape[0]}")
print(f"[W9.4_LOG] Missing y_pred after alignment (filled as 0 for signal): {summary['missing_y_pred_filled_as_0']} / {len(bt_df)}")

print("\n[W9.4_LOG] Distribution checks:")
print("signal_t value counts:")
print(bt_df["signal_t"].value_counts(dropna=False).sort_index())
print("position value counts:")
print(bt_df["position"].value_counts(dropna=False).sort_index())
print(f"[W9.4_LOG] Exposure (position != 0): {summary['exposure_rate']:.6f}")

print("\n[W9.4_LOG] Core metrics (log-core; cost=0):")
for k in [
    "cumulative_log_return","cumulative_simple_return","annualized_vol_ddof0","sharpe_like_rf0_ddof0",
    "max_drawdown","exposure_rate","turnover",
    "n_roundtrip_trades","n_position_change_events","n_flips","win_rate","n_wins","n_losses","n_zero_pnl_trades"
]:
    print(f"{k}: {summary[k]}")

print("\n[W9.4_LOG] Sanity table HEAD (first 8 rows):")
cols = ["date","close","ret_log","y_pred","signal_t","position","strat_ret_log","equity"]
print(bt_df[cols].head(8).to_string(index=False))

print("\n[W9.4_LOG] Sanity table TAIL (last 8 rows):")
print(bt_df[cols].tail(8).to_string(index=False))

print("\n[W9.4_LOG] Trades head (if any):")
print(trades_df.head(10).to_string(index=False) if len(trades_df) else "NO TRADES")


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("..").resolve()
OHLC_PATH = ROOT / "data" / "processed" / "mru_usd_ohlc_clean.csv"

V1_START = pd.Timestamp("2020-01-01")
V1_END   = pd.Timestamp("2025-11-25")

def compute_roundtrip_trades(df: pd.DataFrame):
    pos = df["position"].to_numpy()
    strat = df["strat_ret_log"].to_numpy()

    trades = []
    i = 0
    n = len(df)
    while i < n:
        if pos[i] == 0:
            i += 1
            continue
        direction = pos[i]
        j = i
        while j + 1 < n and pos[j + 1] == direction:
            j += 1
        pnl = float(np.nansum(strat[i:j+1]))
        trades.append({
            "t_in": df.loc[i, "date"],
            "t_out_boundary": (df.loc[j+1, "date"] if j+1 < n else df.loc[j, "date"]),
            "dir": int(direction),
            "pnl_log": pnl
        })
        i = j + 1
    return pd.DataFrame(trades)

def backtest_B1_log_core(window_start: pd.Timestamp, window_end: pd.Timestamp):
    ohlc = pd.read_csv(OHLC_PATH, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
    ohlc["log_close"] = np.log(ohlc["close"].astype(float))
    ohlc["ret_log"] = ohlc["log_close"].diff()

    win = ohlc[(ohlc["date"] >= window_start) & (ohlc["date"] <= window_end)].copy().reset_index(drop=True)

    # B1 "signal" is conceptually +1 always,
    win["signal_t"] = 1
    win["position"] = 1
    win.loc[0, "position"] = 0  # HARD RULE

    # log-core returns, cost=0
    win["strat_ret_log"] = win["position"].astype(float) * win["ret_log"].astype(float)
    win["equity"] = np.exp(win["strat_ret_log"].fillna(0.0).cumsum())

    exposure_rate = float((win["position"] != 0).mean())
    turnover = float(win["position"].diff().abs().fillna(0.0).mean())

    pos_diff = win["position"].diff()
    n_pos_change_events = int((pos_diff.iloc[1:] != 0).sum())
    n_flips = int((pos_diff.abs().iloc[1:] == 2).sum())

    trades = compute_roundtrip_trades(win)
    n_roundtrips = int(len(trades))
    if n_roundtrips == 0:
        win_rate = np.nan
        n_wins = 0; n_losses = 0; n_zero = 0
    else:
        n_wins = int((trades["pnl_log"] > 0).sum())
        n_losses = int((trades["pnl_log"] < 0).sum())
        n_zero = int((trades["pnl_log"] == 0).sum())
        win_rate = n_wins / n_roundtrips

    cum_log = float(np.nansum(win["strat_ret_log"]))
    cum_simple = float(np.exp(cum_log) - 1.0)

    x = win["strat_ret_log"].to_numpy()
    daily_std = float(np.nanstd(x, ddof=0))
    daily_mean = float(np.nanmean(x))
    ann_vol = daily_std * float(np.sqrt(252.0))
    sharpe = (daily_mean / daily_std * float(np.sqrt(252.0))) if daily_std > 0 else np.nan

    eq = win["equity"].astype(float)
    dd = (eq / eq.cummax()) - 1.0
    max_dd = float(dd.min())

    summary = {
        "cumulative_log_return": cum_log,
        "cumulative_simple_return": cum_simple,
        "annualized_vol_ddof0": ann_vol,
        "sharpe_like_rf0_ddof0": sharpe,
        "max_drawdown": max_dd,
        "exposure_rate": exposure_rate,
        "turnover": turnover,
        "n_roundtrip_trades": n_roundtrips,
        "n_position_change_events": n_pos_change_events,
        "n_flips": n_flips,
        "win_rate": win_rate,
        "n_wins": n_wins,
        "n_losses": n_losses,
        "n_zero_pnl_trades": n_zero,
    }
    return win, trades, summary

print(f"[W9.4_LOG] Run: B1_buyhold_V1")
print(f"[W9.4_LOG] Declared window: {V1_START.date()} → {V1_END.date()}")

bt_df, trades_df, summary = backtest_B1_log_core(V1_START, V1_END)

print(f"[W9.4_LOG] OHLC window rows: {len(bt_df)} | OHLC min/max: {bt_df['date'].min().date()} → {bt_df['date'].max().date()}")

print("\n[W9.4_LOG] Distribution checks:")
print("position value counts:")
print(bt_df["position"].value_counts(dropna=False).sort_index())
print(f"[W9.4_LOG] Exposure (position != 0): {summary['exposure_rate']:.6f}")

print("\n[W9.4_LOG] Core metrics (log-core; cost=0):")
for k in [
    "cumulative_log_return","cumulative_simple_return","annualized_vol_ddof0","sharpe_like_rf0_ddof0",
    "max_drawdown","exposure_rate","turnover",
    "n_roundtrip_trades","n_position_change_events","n_flips","win_rate","n_wins","n_losses","n_zero_pnl_trades"
]:
    print(f"{k}: {summary[k]}")

print("\n[W9.4_LOG] Sanity table HEAD (first 8 rows):")
cols = ["date","close","ret_log","position","strat_ret_log","equity"]
print(bt_df[cols].head(8).to_string(index=False))

print("\n[W9.4_LOG] Sanity table TAIL (last 8 rows):")
print(bt_df[cols].tail(8).to_string(index=False))

print("\n[W9.4_LOG] Trades (if any):")
print(trades_df.to_string(index=False) if len(trades_df) else "NO TRADES")


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("..").resolve()
OHLC_PATH = ROOT / "data" / "processed" / "mru_usd_ohlc_clean.csv"
PRED_PATH = ROOT / "results" / "predictions" / "predictions_E7_V2_F1_test.csv"

WIN_START = pd.Timestamp("2010-01-01")
WIN_END   = pd.Timestamp("2011-12-31")

def _detect_col(cols, candidates):
    cols_l = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in cols_l:
            return cols_l[cand.lower()]
    return None

def compute_roundtrip_trades(df: pd.DataFrame):
    pos = df["position"].to_numpy()
    strat = df["strat_ret_log"].to_numpy()

    trades = []
    i = 0
    n = len(df)
    while i < n:
        if pos[i] == 0:
            i += 1
            continue
        direction = pos[i]
        j = i
        while j + 1 < n and pos[j + 1] == direction:
            j += 1
        pnl = float(np.nansum(strat[i:j+1]))
        trades.append({
            "t_in": df.loc[i, "date"],
            "t_out_boundary": (df.loc[j+1, "date"] if j+1 < n else df.loc[j, "date"]),
            "dir": int(direction),
            "pnl_log": pnl
        })
        i = j + 1
    return pd.DataFrame(trades)

def run_backtest_from_predictions(pred_path: Path, window_start: pd.Timestamp, window_end: pd.Timestamp):
    # --- OHLC + canonical log returns
    ohlc = pd.read_csv(OHLC_PATH, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
    ohlc["log_close"] = np.log(ohlc["close"].astype(float))
    ohlc["ret_log"] = ohlc["log_close"].diff()

    win = ohlc[(ohlc["date"] >= window_start) & (ohlc["date"] <= window_end)].copy().reset_index(drop=True)

    # --- Predictions
    pred = pd.read_csv(pred_path)
    date_col = _detect_col(pred.columns, ["date", "Date", "ds", "timestamp"])
    if date_col is None:
        raise ValueError(f"Could not detect date column in {pred_path.name}. Columns: {list(pred.columns)}")
    pred[date_col] = pd.to_datetime(pred[date_col])

    y_col = _detect_col(pred.columns, ["y_pred", "pred", "yhat", "yhat_t", "prediction"])
    if y_col is None:
        raise ValueError(f"Could not detect prediction column in {pred_path.name}. Columns: {list(pred.columns)}")

    pred = pred[[date_col, y_col]].rename(columns={date_col: "date", y_col: "y_pred"}).sort_values("date")
    pred["y_pred"] = pd.to_numeric(pred["y_pred"], errors="coerce")

    # Align (left join onto OHLC window)
    df = win.merge(pred, on="date", how="left")

    missing = int(df["y_pred"].isna().sum())
    df["signal_t"] = df["y_pred"].fillna(0.0)  # missing prediction -> neutral

    # HARD RULE: p_t = f(yhat_{t-1}), and p_start = 0
    df["position"] = df["signal_t"].shift(1).fillna(0.0)
    df.loc[0, "position"] = 0.0

    # log-core strategy return, cost=0
    df["strat_ret_log"] = df["position"].astype(float) * df["ret_log"].astype(float)
    df["equity"] = np.exp(df["strat_ret_log"].fillna(0.0).cumsum())

    exposure_rate = float((df["position"] != 0).mean())
    turnover = float(df["position"].diff().abs().fillna(0.0).mean())

    pos_diff = df["position"].diff()
    n_pos_change_events = int((pos_diff.iloc[1:] != 0).sum())
    n_flips = int((pos_diff.abs().iloc[1:] == 2).sum())

    trades = compute_roundtrip_trades(df)
    n_roundtrips = int(len(trades))
    if n_roundtrips == 0:
        win_rate = np.nan
        n_wins = 0; n_losses = 0; n_zero = 0
    else:
        n_wins = int((trades["pnl_log"] > 0).sum())
        n_losses = int((trades["pnl_log"] < 0).sum())
        n_zero = int((trades["pnl_log"] == 0).sum())
        win_rate = n_wins / n_roundtrips

    cum_log = float(np.nansum(df["strat_ret_log"]))
    cum_simple = float(np.exp(cum_log) - 1.0)

    x = df["strat_ret_log"].to_numpy()
    daily_std = float(np.nanstd(x, ddof=0))
    daily_mean = float(np.nanmean(x))
    ann_vol = daily_std * float(np.sqrt(252.0))
    sharpe = (daily_mean / daily_std * float(np.sqrt(252.0))) if daily_std > 0 else np.nan

    eq = df["equity"].astype(float)
    dd = (eq / eq.cummax()) - 1.0
    max_dd = float(dd.min())

    summary = {
        "cumulative_log_return": cum_log,
        "cumulative_simple_return": cum_simple,
        "annualized_vol_ddof0": ann_vol,
        "sharpe_like_rf0_ddof0": sharpe,
        "max_drawdown": max_dd,
        "exposure_rate": exposure_rate,
        "turnover": turnover,
        "n_roundtrip_trades": n_roundtrips,
        "n_position_change_events": n_pos_change_events,
        "n_flips": n_flips,
        "win_rate": win_rate,
        "n_wins": n_wins,
        "n_losses": n_losses,
        "n_zero_pnl_trades": n_zero,
        "missing_y_pred_filled_neutral": missing,
    }
    return df, trades, summary

print(f"[W9.4_LOG] Run: T2_L2_V2_F1 (E7)")
print(f"[W9.4_LOG] Declared window: {WIN_START.date()} → {WIN_END.date()}")

bt_df, trades_df, summary = run_backtest_from_predictions(PRED_PATH, WIN_START, WIN_END)

print(f"[W9.4_LOG] OHLC window rows: {len(bt_df)} | OHLC min/max: {bt_df['date'].min().date()} → {bt_df['date'].max().date()}")
print(f"[W9.4_LOG] Prediction file rows: {len(pd.read_csv(PRED_PATH))}")
print(f"[W9.4_LOG] Missing y_pred after alignment (filled as 0 for signal): {summary['missing_y_pred_filled_neutral']} / {len(bt_df)}")

print("\n[W9.4_LOG] Distribution checks:")
print("signal_t value counts:")
print(pd.Series(bt_df["signal_t"]).value_counts(dropna=False).sort_index())
print("position value counts:")
print(pd.Series(bt_df["position"]).value_counts(dropna=False).sort_index())
print(f"[W9.4_LOG] Exposure (position != 0): {summary['exposure_rate']:.6f}")

print("\n[W9.4_LOG] Core metrics (log-core; cost=0):")
for k in [
    "cumulative_log_return","cumulative_simple_return","annualized_vol_ddof0","sharpe_like_rf0_ddof0",
    "max_drawdown","exposure_rate","turnover",
    "n_roundtrip_trades","n_position_change_events","n_flips","win_rate","n_wins","n_losses","n_zero_pnl_trades"
]:
    print(f"{k}: {summary[k]}")

print("\n[W9.4_LOG] Sanity table HEAD (first 8 rows):")
cols = ["date","close","ret_log","y_pred","signal_t","position","strat_ret_log","equity"]
print(bt_df[cols].head(8).to_string(index=False))

print("\n[W9.4_LOG] Sanity table TAIL (last 8 rows):")
print(bt_df[cols].tail(8).to_string(index=False))

print("\n[W9.4_LOG] Trades head (if any):")
print(trades_df.head(10).to_string(index=False) if len(trades_df) else "NO TRADES")


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("..").resolve()
OHLC_PATH = ROOT / "data" / "processed" / "mru_usd_ohlc_clean.csv"
PRED_PATH = ROOT / "results" / "predictions" / "predictions_E7_V2_F2_test.csv"

WIN_START = pd.Timestamp("2012-01-01")
WIN_END   = pd.Timestamp("2013-12-31")

def _detect_col(cols, candidates):
    cols_l = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in cols_l:
            return cols_l[cand.lower()]
    return None

def compute_roundtrip_trades(df: pd.DataFrame):
    pos = df["position"].to_numpy()
    strat = df["strat_ret_log"].to_numpy()

    trades = []
    i = 0
    n = len(df)
    while i < n:
        if pos[i] == 0:
            i += 1
            continue
        direction = pos[i]
        j = i
        while j + 1 < n and pos[j + 1] == direction:
            j += 1
        pnl = float(np.nansum(strat[i:j+1]))
        trades.append({
            "t_in": df.loc[i, "date"],
            "t_out_boundary": (df.loc[j+1, "date"] if j+1 < n else df.loc[j, "date"]),
            "dir": int(direction),
            "pnl_log": pnl
        })
        i = j + 1
    return pd.DataFrame(trades)

def run_backtest_from_predictions(pred_path: Path, window_start: pd.Timestamp, window_end: pd.Timestamp):
    ohlc = pd.read_csv(OHLC_PATH, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
    ohlc["log_close"] = np.log(ohlc["close"].astype(float))
    ohlc["ret_log"] = ohlc["log_close"].diff()

    win = ohlc[(ohlc["date"] >= window_start) & (ohlc["date"] <= window_end)].copy().reset_index(drop=True)

    pred = pd.read_csv(pred_path)
    date_col = _detect_col(pred.columns, ["date", "Date", "ds", "timestamp"])
    if date_col is None:
        raise ValueError(f"Could not detect date column in {pred_path.name}. Columns: {list(pred.columns)}")
    pred[date_col] = pd.to_datetime(pred[date_col])

    y_col = _detect_col(pred.columns, ["y_pred", "pred", "yhat", "yhat_t", "prediction"])
    if y_col is None:
        raise ValueError(f"Could not detect prediction column in {pred_path.name}. Columns: {list(pred.columns)}")

    pred = pred[[date_col, y_col]].rename(columns={date_col: "date", y_col: "y_pred"}).sort_values("date")
    pred["y_pred"] = pd.to_numeric(pred["y_pred"], errors="coerce")

    df = win.merge(pred, on="date", how="left")

    missing = int(df["y_pred"].isna().sum())
    df["signal_t"] = df["y_pred"].fillna(0.0)  

    df["position"] = df["signal_t"].shift(1).fillna(0.0)
    df.loc[0, "position"] = 0.0

    df["strat_ret_log"] = df["position"].astype(float) * df["ret_log"].astype(float)
    df["equity"] = np.exp(df["strat_ret_log"].fillna(0.0).cumsum())

    exposure_rate = float((df["position"] != 0).mean())
    turnover = float(df["position"].diff().abs().fillna(0.0).mean())

    pos_diff = df["position"].diff()
    n_pos_change_events = int((pos_diff.iloc[1:] != 0).sum())
    n_flips = int((pos_diff.abs().iloc[1:] == 2).sum())

    trades = compute_roundtrip_trades(df)
    n_roundtrips = int(len(trades))
    if n_roundtrips == 0:
        win_rate = np.nan
        n_wins = 0; n_losses = 0; n_zero = 0
    else:
        n_wins = int((trades["pnl_log"] > 0).sum())
        n_losses = int((trades["pnl_log"] < 0).sum())
        n_zero = int((trades["pnl_log"] == 0).sum())
        win_rate = n_wins / n_roundtrips

    cum_log = float(np.nansum(df["strat_ret_log"]))
    cum_simple = float(np.exp(cum_log) - 1.0)

    x = df["strat_ret_log"].to_numpy()
    daily_std = float(np.nanstd(x, ddof=0))
    daily_mean = float(np.nanmean(x))
    ann_vol = daily_std * float(np.sqrt(252.0))
    sharpe = (daily_mean / daily_std * float(np.sqrt(252.0))) if daily_std > 0 else np.nan

    eq = df["equity"].astype(float)
    dd = (eq / eq.cummax()) - 1.0
    max_dd = float(dd.min())

    summary = {
        "cumulative_log_return": cum_log,
        "cumulative_simple_return": cum_simple,
        "annualized_vol_ddof0": ann_vol,
        "sharpe_like_rf0_ddof0": sharpe,
        "max_drawdown": max_dd,
        "exposure_rate": exposure_rate,
        "turnover": turnover,
        "n_roundtrip_trades": n_roundtrips,
        "n_position_change_events": n_pos_change_events,
        "n_flips": n_flips,
        "win_rate": win_rate,
        "n_wins": n_wins,
        "n_losses": n_losses,
        "n_zero_pnl_trades": n_zero,
        "missing_y_pred_filled_neutral": missing,
    }
    return df, trades, summary

print(f"[W9.4_LOG] Run: T2_L2_V2_F2 (E7)")
print(f"[W9.4_LOG] Declared window: {WIN_START.date()} → {WIN_END.date()}")

bt_df, trades_df, summary = run_backtest_from_predictions(PRED_PATH, WIN_START, WIN_END)

print(f"[W9.4_LOG] OHLC window rows: {len(bt_df)} | OHLC min/max: {bt_df['date'].min().date()} → {bt_df['date'].max().date()}")
print(f"[W9.4_LOG] Prediction file rows: {len(pd.read_csv(PRED_PATH))}")
print(f"[W9.4_LOG] Missing y_pred after alignment (filled as 0 for signal): {summary['missing_y_pred_filled_neutral']} / {len(bt_df)}")

print("\n[W9.4_LOG] Distribution checks:")
print("signal_t value counts:")
print(pd.Series(bt_df["signal_t"]).value_counts(dropna=False).sort_index())
print("position value counts:")
print(pd.Series(bt_df["position"]).value_counts(dropna=False).sort_index())
print(f"[W9.4_LOG] Exposure (position != 0): {summary['exposure_rate']:.6f}")

print("\n[W9.4_LOG] Core metrics (log-core; cost=0):")
for k in [
    "cumulative_log_return","cumulative_simple_return","annualized_vol_ddof0","sharpe_like_rf0_ddof0",
    "max_drawdown","exposure_rate","turnover",
    "n_roundtrip_trades","n_position_change_events","n_flips","win_rate","n_wins","n_losses","n_zero_pnl_trades"
]:
    print(f"{k}: {summary[k]}")

print("\n[W9.4_LOG] Sanity table HEAD (first 8 rows):")
cols = ["date","close","ret_log","y_pred","signal_t","position","strat_ret_log","equity"]
print(bt_df[cols].head(8).to_string(index=False))

print("\n[W9.4_LOG] Sanity table TAIL (last 8 rows):")
print(bt_df[cols].tail(8).to_string(index=False))

print("\n[W9.4_LOG] Trades head (if any):")
print(trades_df.head(10).to_string(index=False) if len(trades_df) else "NO TRADES")


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("..").resolve()
OHLC_PATH = ROOT / "data" / "processed" / "mru_usd_ohlc_clean.csv"
PRED_PATH = ROOT / "results" / "predictions" / "predictions_E7_V2_F3_test.csv"

WIN_START = pd.Timestamp("2014-01-01")
WIN_END   = pd.Timestamp("2015-12-31")

def _detect_col(cols, candidates):
    cols_l = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in cols_l:
            return cols_l[cand.lower()]
    return None

def compute_roundtrip_trades(df: pd.DataFrame):
    pos = df["position"].to_numpy()
    strat = df["strat_ret_log"].to_numpy()

    trades = []
    i = 0
    n = len(df)
    while i < n:
        if pos[i] == 0:
            i += 1
            continue
        direction = pos[i]
        j = i
        while j + 1 < n and pos[j + 1] == direction:
            j += 1
        pnl = float(np.nansum(strat[i:j+1]))
        trades.append({
            "t_in": df.loc[i, "date"],
            "t_out_boundary": (df.loc[j+1, "date"] if j+1 < n else df.loc[j, "date"]),
            "dir": int(direction),
            "pnl_log": pnl
        })
        i = j + 1
    return pd.DataFrame(trades)

def run_backtest_from_predictions(pred_path: Path, window_start: pd.Timestamp, window_end: pd.Timestamp):
    ohlc = pd.read_csv(OHLC_PATH, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
    ohlc["log_close"] = np.log(ohlc["close"].astype(float))
    ohlc["ret_log"] = ohlc["log_close"].diff()

    win = ohlc[(ohlc["date"] >= window_start) & (ohlc["date"] <= window_end)].copy().reset_index(drop=True)

    pred = pd.read_csv(pred_path)
    date_col = _detect_col(pred.columns, ["date", "Date", "ds", "timestamp"])
    if date_col is None:
        raise ValueError(f"Could not detect date column in {pred_path.name}. Columns: {list(pred.columns)}")
    pred[date_col] = pd.to_datetime(pred[date_col])

    y_col = _detect_col(pred.columns, ["y_pred", "pred", "yhat", "yhat_t", "prediction"])
    if y_col is None:
        raise ValueError(f"Could not detect prediction column in {pred_path.name}. Columns: {list(pred.columns)}")

    pred = pred[[date_col, y_col]].rename(columns={date_col: "date", y_col: "y_pred"}).sort_values("date")
    pred["y_pred"] = pd.to_numeric(pred["y_pred"], errors="coerce")

    df = win.merge(pred, on="date", how="left")

    missing = int(df["y_pred"].isna().sum())
    df["signal_t"] = df["y_pred"].fillna(0.0)
    df["position"] = df["signal_t"].shift(1).fillna(0.0)
    df.loc[0, "position"] = 0.0

    df["strat_ret_log"] = df["position"].astype(float) * df["ret_log"].astype(float)
    df["equity"] = np.exp(df["strat_ret_log"].fillna(0.0).cumsum())

    exposure_rate = float((df["position"] != 0).mean())
    turnover = float(df["position"].diff().abs().fillna(0.0).mean())

    pos_diff = df["position"].diff()
    n_pos_change_events = int((pos_diff.iloc[1:] != 0).sum())
    n_flips = int((pos_diff.abs().iloc[1:] == 2).sum())

    trades = compute_roundtrip_trades(df)
    n_roundtrips = int(len(trades))
    if n_roundtrips == 0:
        win_rate = np.nan
        n_wins = 0; n_losses = 0; n_zero = 0
    else:
        n_wins = int((trades["pnl_log"] > 0).sum())
        n_losses = int((trades["pnl_log"] < 0).sum())
        n_zero = int((trades["pnl_log"] == 0).sum())
        win_rate = n_wins / n_roundtrips

    cum_log = float(np.nansum(df["strat_ret_log"]))
    cum_simple = float(np.exp(cum_log) - 1.0)

    x = df["strat_ret_log"].to_numpy()
    daily_std = float(np.nanstd(x, ddof=0))
    daily_mean = float(np.nanmean(x))
    ann_vol = daily_std * float(np.sqrt(252.0))
    sharpe = (daily_mean / daily_std * float(np.sqrt(252.0))) if daily_std > 0 else np.nan

    eq = df["equity"].astype(float)
    dd = (eq / eq.cummax()) - 1.0
    max_dd = float(dd.min())

    summary = {
        "cumulative_log_return": cum_log,
        "cumulative_simple_return": cum_simple,
        "annualized_vol_ddof0": ann_vol,
        "sharpe_like_rf0_ddof0": sharpe,
        "max_drawdown": max_dd,
        "exposure_rate": exposure_rate,
        "turnover": turnover,
        "n_roundtrip_trades": n_roundtrips,
        "n_position_change_events": n_pos_change_events,
        "n_flips": n_flips,
        "win_rate": win_rate,
        "n_wins": n_wins,
        "n_losses": n_losses,
        "n_zero_pnl_trades": n_zero,
        "missing_y_pred_filled_neutral": missing,
    }
    return df, trades, summary

print(f"[W9.4_LOG] Run: T2_L2_V2_F3 (E7)")
print(f"[W9.4_LOG] Declared window: {WIN_START.date()} → {WIN_END.date()}")

bt_df, trades_df, summary = run_backtest_from_predictions(PRED_PATH, WIN_START, WIN_END)

print(f"[W9.4_LOG] OHLC window rows: {len(bt_df)} | OHLC min/max: {bt_df['date'].min().date()} → {bt_df['date'].max().date()}")
print(f"[W9.4_LOG] Prediction file rows: {len(pd.read_csv(PRED_PATH))}")
print(f"[W9.4_LOG] Missing y_pred after alignment (filled as 0 for signal): {summary['missing_y_pred_filled_neutral']} / {len(bt_df)}")

print("\n[W9.4_LOG] Distribution checks:")
print("signal_t value counts:")
print(pd.Series(bt_df["signal_t"]).value_counts(dropna=False).sort_index())
print("position value counts:")
print(pd.Series(bt_df["position"]).value_counts(dropna=False).sort_index())
print(f"[W9.4_LOG] Exposure (position != 0): {summary['exposure_rate']:.6f}")

print("\n[W9.4_LOG] Core metrics (log-core; cost=0):")
for k in [
    "cumulative_log_return","cumulative_simple_return","annualized_vol_ddof0","sharpe_like_rf0_ddof0",
    "max_drawdown","exposure_rate","turnover",
    "n_roundtrip_trades","n_position_change_events","n_flips","win_rate","n_wins","n_losses","n_zero_pnl_trades"
]:
    print(f"{k}: {summary[k]}")

print("\n[W9.4_LOG] Sanity table HEAD (first 8 rows):")
cols = ["date","close","ret_log","y_pred","signal_t","position","strat_ret_log","equity"]
print(bt_df[cols].head(8).to_string(index=False))

print("\n[W9.4_LOG] Sanity table TAIL (last 8 rows):")
print(bt_df[cols].tail(8).to_string(index=False))

print("\n[W9.4_LOG] Trades head (if any):")
print(trades_df.head(10).to_string(index=False) if len(trades_df) else "NO TRADES")


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("..").resolve()
OHLC_PATH = ROOT / "data" / "processed" / "mru_usd_ohlc_clean.csv"
PRED_PATH = ROOT / "results" / "predictions" / "predictions_E7_V2_F4_test.csv"

WIN_START = pd.Timestamp("2016-01-01")
WIN_END   = pd.Timestamp("2019-12-31")

def _detect_col(cols, candidates):
    cols_l = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in cols_l:
            return cols_l[cand.lower()]
    return None

def compute_roundtrip_trades(df: pd.DataFrame):
    pos = df["position"].to_numpy()
    strat = df["strat_ret_log"].to_numpy()

    trades = []
    i = 0
    n = len(df)
    while i < n:
        if pos[i] == 0:
            i += 1
            continue
        direction = pos[i]
        j = i
        while j + 1 < n and pos[j + 1] == direction:
            j += 1
        pnl = float(np.nansum(strat[i:j+1]))
        trades.append({
            "t_in": df.loc[i, "date"],
            "t_out_boundary": (df.loc[j+1, "date"] if j+1 < n else df.loc[j, "date"]),
            "dir": int(direction),
            "pnl_log": pnl
        })
        i = j + 1
    return pd.DataFrame(trades)

def run_backtest_from_predictions(pred_path: Path, window_start: pd.Timestamp, window_end: pd.Timestamp):
    ohlc = pd.read_csv(OHLC_PATH, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
    ohlc["log_close"] = np.log(ohlc["close"].astype(float))
    ohlc["ret_log"] = ohlc["log_close"].diff()

    win = ohlc[(ohlc["date"] >= window_start) & (ohlc["date"] <= window_end)].copy().reset_index(drop=True)

    pred = pd.read_csv(pred_path)
    date_col = _detect_col(pred.columns, ["date", "Date", "ds", "timestamp"])
    if date_col is None:
        raise ValueError(f"Could not detect date column in {pred_path.name}. Columns: {list(pred.columns)}")
    pred[date_col] = pd.to_datetime(pred[date_col])

    y_col = _detect_col(pred.columns, ["y_pred", "pred", "yhat", "yhat_t", "prediction"])
    if y_col is None:
        raise ValueError(f"Could not detect prediction column in {pred_path.name}. Columns: {list(pred.columns)}")

    pred = pred[[date_col, y_col]].rename(columns={date_col: "date", y_col: "y_pred"}).sort_values("date")
    pred["y_pred"] = pd.to_numeric(pred["y_pred"], errors="coerce")

    df = win.merge(pred, on="date", how="left")

    missing = int(df["y_pred"].isna().sum())
    df["signal_t"] = df["y_pred"].fillna(0.0)
    df["position"] = df["signal_t"].shift(1).fillna(0.0)
    df.loc[0, "position"] = 0.0

    df["strat_ret_log"] = df["position"].astype(float) * df["ret_log"].astype(float)
    df["equity"] = np.exp(df["strat_ret_log"].fillna(0.0).cumsum())

    exposure_rate = float((df["position"] != 0).mean())
    turnover = float(df["position"].diff().abs().fillna(0.0).mean())

    pos_diff = df["position"].diff()
    n_pos_change_events = int((pos_diff.iloc[1:] != 0).sum())
    n_flips = int((pos_diff.abs().iloc[1:] == 2).sum())

    trades = compute_roundtrip_trades(df)
    n_roundtrips = int(len(trades))
    if n_roundtrips == 0:
        win_rate = np.nan
        n_wins = 0; n_losses = 0; n_zero = 0
    else:
        n_wins = int((trades["pnl_log"] > 0).sum())
        n_losses = int((trades["pnl_log"] < 0).sum())
        n_zero = int((trades["pnl_log"] == 0).sum())
        win_rate = n_wins / n_roundtrips

    cum_log = float(np.nansum(df["strat_ret_log"]))
    cum_simple = float(np.exp(cum_log) - 1.0)

    x = df["strat_ret_log"].to_numpy()
    daily_std = float(np.nanstd(x, ddof=0))
    daily_mean = float(np.nanmean(x))
    ann_vol = daily_std * float(np.sqrt(252.0))
    sharpe = (daily_mean / daily_std * float(np.sqrt(252.0))) if daily_std > 0 else np.nan

    eq = df["equity"].astype(float)
    dd = (eq / eq.cummax()) - 1.0
    max_dd = float(dd.min())

    summary = {
        "cumulative_log_return": cum_log,
        "cumulative_simple_return": cum_simple,
        "annualized_vol_ddof0": ann_vol,
        "sharpe_like_rf0_ddof0": sharpe,
        "max_drawdown": max_dd,
        "exposure_rate": exposure_rate,
        "turnover": turnover,
        "n_roundtrip_trades": n_roundtrips,
        "n_position_change_events": n_pos_change_events,
        "n_flips": n_flips,
        "win_rate": win_rate,
        "n_wins": n_wins,
        "n_losses": n_losses,
        "n_zero_pnl_trades": n_zero,
        "missing_y_pred_filled_neutral": missing,
    }
    return df, trades, summary

print(f"[W9.4_LOG] Run: T2_L2_V2_F4 (E7)")
print(f"[W9.4_LOG] Declared window: {WIN_START.date()} → {WIN_END.date()}")

bt_df, trades_df, summary = run_backtest_from_predictions(PRED_PATH, WIN_START, WIN_END)

print(f"[W9.4_LOG] OHLC window rows: {len(bt_df)} | OHLC min/max: {bt_df['date'].min().date()} → {bt_df['date'].max().date()}")
print(f"[W9.4_LOG] Prediction file rows: {len(pd.read_csv(PRED_PATH))}")
print(f"[W9.4_LOG] Missing y_pred after alignment (filled as 0 for signal): {summary['missing_y_pred_filled_neutral']} / {len(bt_df)}")

print("\n[W9.4_LOG] Distribution checks:")
print("signal_t value counts:")
print(pd.Series(bt_df["signal_t"]).value_counts(dropna=False).sort_index())
print("position value counts:")
print(pd.Series(bt_df["position"]).value_counts(dropna=False).sort_index())
print(f"[W9.4_LOG] Exposure (position != 0): {summary['exposure_rate']:.6f}")

print("\n[W9.4_LOG] Core metrics (log-core; cost=0):")
for k in [
    "cumulative_log_return","cumulative_simple_return","annualized_vol_ddof0","sharpe_like_rf0_ddof0",
    "max_drawdown","exposure_rate","turnover",
    "n_roundtrip_trades","n_position_change_events","n_flips","win_rate","n_wins","n_losses","n_zero_pnl_trades"
]:
    print(f"{k}: {summary[k]}")

print("\n[W9.4_LOG] Sanity table HEAD (first 8 rows):")
cols = ["date","close","ret_log","y_pred","signal_t","position","strat_ret_log","equity"]
print(bt_df[cols].head(8).to_string(index=False))

print("\n[W9.4_LOG] Sanity table TAIL (last 8 rows):")
print(bt_df[cols].tail(8).to_string(index=False))

print("\n[W9.4_LOG] Trades head (if any):")
print(trades_df.head(10).to_string(index=False) if len(trades_df) else "NO TRADES")


+++++++++++++++++++++++


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("..").resolve()
pred_path = ROOT / "results" / "predictions" / "predictions_E1_V1_test.csv"

df = pd.read_csv(pred_path)
print("[SCHEMA] file:", pred_path)
print("[SCHEMA] columns:", list(df.columns))
print("[SCHEMA] dtypes:\n", df.dtypes)

# date parsing check
date_col_candidates = [c for c in df.columns if c.lower() in ("date", "datetime", "dt")]
print("[SCHEMA] date_col_candidates:", date_col_candidates)

# predicted-label candidates (try to find the discrete -1/0/+1 column)
cand = []
for c in df.columns:
    cl = c.lower()
    if any(k in cl for k in ["y_pred", "yhat", "pred", "prediction"]) and "proba" not in cl and "prob" not in cl:
        cand.append(c)
print("[SCHEMA] yhat candidates:", cand)

print("\n[HEAD]\n", df.head(5))
print("\n[TAIL]\n", df.tail(5))

if len(cand) == 1:
    c = cand[0]
    print(f"\n[VALUE_COUNTS] {c}:\n", df[c].value_counts(dropna=False).sort_index())
else:
    print("\n[STOP] Multiple/zero yhat candidates.")


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 200)

ROOT = Path("..").resolve()
pred_path = ROOT / "results" / "predictions" / "predictions_E1_V1_test.csv"

print("[E1_CHECK] Project ROOT:", ROOT)
print("[E1_CHECK] Target file:", pred_path)
print("[E1_CHECK] Exists:", pred_path.exists())
assert pred_path.exists(), f"Missing file: {pred_path}"


In [ ]:
def guess_date_col(columns):
    cols = list(columns)
    lower = {c: c.lower() for c in cols}
    # strong candidates
    priority = ["date", "datetime", "timestamp", "dt", "time"]
    exact = [c for c in cols if lower[c] in priority]
    if exact:
        return exact[0], "exact_match"
    # contains patterns
    contains = [c for c in cols if any(k in lower[c] for k in ["date", "time", "dt"])]
    if contains:
        return contains[0], "contains_match"
    return None, "none"

date_col, how = guess_date_col(df.columns)
print("[E1_DATE] guessed date_col:", date_col, "| method:", how)

if date_col is None:
    print("[E1_DATE] STOP: No obvious date column.")
else:
    tmp = df.copy()
    tmp[date_col] = pd.to_datetime(tmp[date_col], errors="coerce")

    n_bad = tmp[date_col].isna().sum()
    print("[E1_DATE] parse NaT count:", int(n_bad))

    if n_bad == len(tmp):
        print("[E1_DATE] STOP: date parsing failed for all rows.")
    else:
        dmin = tmp[date_col].min()
        dmax = tmp[date_col].max()
        print("[E1_DATE] min/max:", dmin, "→", dmax)

        # duplicates + monotonicity
        dup = tmp[date_col].duplicated().sum()
        print("[E1_DATE] duplicated dates:", int(dup))
        mono = tmp[date_col].is_monotonic_increasing
        print("[E1_DATE] is_monotonic_increasing:", bool(mono))

        # show any missing date rows (first few)
        if n_bad > 0:
            bad_rows = tmp[tmp[date_col].isna()].head(5)
            print("\n[E1_DATE] rows with NaT (first 5):\n", bad_rows)


In [ ]:
def pred_col_candidates(df):
    cols = list(df.columns)
    lc = {c: c.lower() for c in cols}

    cand = []
    for c in cols:
        name = lc[c]
        if any(k in name for k in ["y_pred", "yhat", "pred", "prediction", "class"]):
            if not any(p in name for p in ["proba", "prob", "score", "logit"]):
                cand.append(c)

    numeric = [c for c in cols if pd.api.types.is_numeric_dtype(df[c])]
    return cand, numeric

cand, numeric = pred_col_candidates(df)
print("[E1_PRED] named prediction candidates:", cand)
print("[E1_PRED] numeric columns:", numeric)

def summarize_col(s, colname):
    s_num = pd.to_numeric(s, errors="coerce")
    out = {}
    out["dtype"] = str(s.dtype)
    out["n"] = int(len(s))
    out["n_nan"] = int(s_num.isna().sum()) if pd.api.types.is_numeric_dtype(s_num) else int(pd.isna(s).sum())
    out["n_unique_non_nan"] = int(pd.Series(s).dropna().nunique())

    if pd.api.types.is_numeric_dtype(s_num):
        vals = s_num.dropna()
        out["min"] = float(vals.min()) if len(vals) else None
        out["max"] = float(vals.max()) if len(vals) else None
        out["unique_sample"] = sorted(vals.unique()[:10].tolist()) if len(vals) else []
        # check if values look like {-1,0,1}
        allowed = set([-1, 0, 1])
        uniq = set(vals.unique().tolist())
        out["is_subset_of_-1_0_1"] = uniq.issubset(allowed)
        out["near_int_all"] = bool(np.allclose(vals, np.round(vals))) if len(vals) else None
        out["value_counts_top"] = s_num.value_counts(dropna=False).head(12)
    else:
        out["value_counts_top"] = pd.Series(s).value_counts(dropna=False).head(12)

    print(f"\n--- [COL] {colname} ---")
    for k in ["dtype","n","n_nan","n_unique_non_nan","min","max","is_subset_of_-1_0_1","near_int_all"]:
        if k in out:
            print(f"{k}: {out[k]}")
    print("value_counts_top:\n", out["value_counts_top"])

targets = cand if len(cand) > 0 else numeric
if len(targets) == 0:
    print("[E1_PRED] STOP: No candidate columns found ")
else:
    for c in targets:
        summarize_col(df[c], c)


In [ ]:
def is_discrete_minus1_0_1(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if len(s) == 0:
        return False
    uniq = set(s.unique().tolist())
    return uniq.issubset(set([-1, 0, 1]))

discrete_cols = [c for c in df.columns if is_discrete_minus1_0_1(df[c])]
print("[E1_AUTO] discrete {-1,0,1} columns:", discrete_cols)

if len(discrete_cols) == 1:
    c = discrete_cols[0]
    print(f"[E1_AUTO] UNIQUE DISCRETE COLUMN FOUND: {c}")
    print(df[c].value_counts(dropna=False).sort_index())
elif len(discrete_cols) == 0:
    print("[E1_AUTO] No column is strictly in {-1,0,1}.")

else:
    print("[E1_AUTO] Multiple discrete columns found.")
    for c in discrete_cols:
        print(f"\n[candidate] {c}\n", df[c].value_counts(dropna=False).sort_index())


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("..").resolve()
p = ROOT / "results" / "predictions" / "predictions_E1_V1_test.csv"
df = pd.read_csv(p)

df["date"] = pd.to_datetime(df["date"], errors="raise")
print("[E1] rows:", len(df), "| date:", df["date"].min(), "→", df["date"].max())
print("\n[E1] y_true counts:\n", df["y_true"].value_counts().sort_index())
print("\n[E1] y_pred counts:\n", df["y_pred"].value_counts().sort_index())

# confusion matrix in fixed label order
labels = [-1, 0, 1]
cm = pd.crosstab(df["y_true"], df["y_pred"], rownames=["y_true"], colnames=["y_pred"], dropna=False)
cm = cm.reindex(index=labels, columns=labels, fill_value=0)
print("\n[E1] Confusion matrix (rows=true, cols=pred) labels [-1,0,1]:\n", cm)

acc = (df["y_true"] == df["y_pred"]).mean()
print("\n[E1] Accuracy:", acc)


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("..").resolve()
pred_dir = ROOT / "results" / "predictions"

files = [
    "predictions_E1_V1_test.csv",
    "predictions_E2_V1_test.csv",
    "predictions_E3_V1_test.csv",
    "predictions_E4_V1_test.csv",
]

rows = []
for fn in files:
    p = pred_dir / fn
    df = pd.read_csv(p)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    vc = df["y_pred"].value_counts(dropna=False).to_dict()
    rows.append({
        "file": fn,
        "n": len(df),
        "date_min": df["date"].min(),
        "date_max": df["date"].max(),
        "y_pred_unique": df["y_pred"].nunique(dropna=False),
        "y_pred_counts": vc,
    })

out = pd.DataFrame(rows)
print(out.to_string(index=False))


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("..").resolve()
pred_dir = ROOT / "results" / "predictions"

def vc(fn):
    df = pd.read_csv(pred_dir / fn)
    return df["y_pred"].value_counts().reindex([-1,0,1], fill_value=0)

table = pd.DataFrame({
    "E1_V1": vc("predictions_E1_V1_test.csv"),
    "E2_V1": vc("predictions_E2_V1_test.csv"),
    "E3_V1": vc("predictions_E3_V1_test.csv"),
    "E4_V1": vc("predictions_E4_V1_test.csv"),
})
table.index.name = "y_pred"
print(table)


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("..").resolve()
pred_dir = ROOT / "results" / "predictions"

rows = []
for p in sorted(pred_dir.glob("predictions_*.csv")):
    df = pd.read_csv(p)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    vc = df["y_pred"].value_counts(dropna=False).to_dict()
    rows.append({
        "file": str(p.relative_to(ROOT)).replace("\\","/"),
        "n_rows": len(df),
        "date_min": df["date"].min(),
        "date_max": df["date"].max(),
        "y_pred_unique": df["y_pred"].nunique(dropna=False),
        "pct_pred_0": float((df["y_pred"] == 0).mean()) if "y_pred" in df else np.nan,
        "counts": vc,
    })

out = pd.DataFrame(rows).sort_values(["y_pred_unique","file"])
print(out.to_string(index=False))


In [ ]:
from pathlib import Path
import re

ROOT = Path("..").resolve()
targets = [
    "predictions_E1_V1_test.csv",
    "predictions_E2_V1_test.csv",
    "predictions_E3_V1_test.csv",
    "predictions_E4_V1_test.csv",
    "results/predictions",
    "to_csv",
    "y_pred",
    "E1",
    "E2",
    "E3",
    "E4",
]

# files to scan
exts = {".py", ".ipynb", ".md", ".txt", ".yaml", ".yml"}
cands = []
for p in ROOT.rglob("*"):
    if p.is_file() and p.suffix.lower() in exts:
        if any(part in {".venv", "data", "results"} for part in p.parts):
            continue
        cands.append(p)

hits = []
for p in cands:
    try:
        txt = p.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue
    score = sum(1 for t in targets if t in txt)
    if score >= 2:
        hits.append((score, p))

hits = sorted(hits, key=lambda x: (-x[0], str(x[1])))

print(f"Found {len(hits)} candidate files. Showing top 30:\n")
for score, p in hits[:30]:
    print(f"[score={score}] {p.relative_to(ROOT)}")


+++++++++++++++++++++++++++++++++++++++++++++++++++

In [ ]:
# W9 check: run backtest-style diagnostics for E6 (V1) and E8 (V2 folds)
from pathlib import Path
import numpy as np
import pandas as pd

LABEL_ORDER = [-1, 0, 1]
ANNUALIZATION = 252

def _load_ohlc(root: Path) -> pd.DataFrame:
    p = root / "data" / "processed" / "mru_usd_ohlc_clean.csv"
    if not p.exists():
        raise FileNotFoundError(f"Missing OHLC file: {p}")
    df = pd.read_csv(p)
    if "date" not in df.columns or "close" not in df.columns:
        raise ValueError(f"OHLC must include columns ['date','close']; got: {list(df.columns)}")
    df["date"] = pd.to_datetime(df["date"], errors="raise")
    df = df.sort_values("date").drop_duplicates("date")
    df["ret_log"] = np.log(df["close"]).diff().fillna(0.0)
    return df[["date", "close", "ret_log"]]

def _load_pred(p: Path) -> pd.DataFrame:
    if not p.exists():
        raise FileNotFoundError(f"Missing prediction file: {p}")
    df = pd.read_csv(p)
    for c in ["date", "y_true", "y_pred"]:
        if c not in df.columns:
            raise ValueError(f"{p.name} missing required column '{c}'. cols={list(df.columns)}")
    df["date"] = pd.to_datetime(df["date"], errors="raise")
    df = df.sort_values("date").drop_duplicates("date")
    return df[["date", "y_true", "y_pred"]]

def _confmat(df_pred: pd.DataFrame) -> pd.DataFrame:
    cm = pd.crosstab(
        df_pred["y_true"],
        df_pred["y_pred"],
        rownames=["y_true"],
        colnames=["y_pred"],
        dropna=False,
    )
    # enforce fixed label order [-1,0,1]
    cm = cm.reindex(index=LABEL_ORDER, columns=LABEL_ORDER, fill_value=0)
    return cm

def _run_backtest(ohlc_win: pd.DataFrame, pred: pd.DataFrame, fill_missing_pred_with: int = 0) -> dict:
    # align predictions to OHLC window dates
    df = ohlc_win.merge(pred, on="date", how="left")
    missing_pred = int(df["y_pred"].isna().sum())
    df["y_pred"] = df["y_pred"].fillna(fill_missing_pred_with).astype(int)

    # signal is prediction itself; position is next-day execution (shift by 1)
    df["signal_t"] = df["y_pred"].astype(int)
    df["position"] = df["signal_t"].shift(1).fillna(0).astype(int)

    # strategy daily log return
    df["strat_ret_log"] = df["position"] * df["ret_log"]
    df["equity"] = np.exp(df["strat_ret_log"].cumsum())

    # metrics
    cum_log = float(df["strat_ret_log"].sum())
    cum_simple = float(df["equity"].iloc[-1] - 1.0)
    vol_daily = float(df["strat_ret_log"].std(ddof=0))
    vol_annual = float(vol_daily * np.sqrt(ANNUALIZATION))
    mean_daily = float(df["strat_ret_log"].mean())
    sharpe_like = float((mean_daily / vol_daily) * np.sqrt(ANNUALIZATION)) if vol_daily > 0 else np.nan

    peak = df["equity"].cummax()
    dd = df["equity"] / peak - 1.0
    max_dd = float(dd.min())

    pos = df["position"].to_numpy()
    dpos = np.diff(pos, prepend=pos[0])
    n_change_events = int(np.sum(dpos != 0))
    turnover = float(np.sum(np.abs(dpos)) / len(pos))
    exposure = float(np.mean(pos != 0))

    flips = 0
    prev = 0
    for v in pos:
        if v == 0:
            continue
        if prev != 0 and np.sign(v) != np.sign(prev):
            flips += 1
        prev = v
    flips = int(flips)


    segments = []
    in_trade = False
    t_in = None
    cur_dir = None
    for i, (dt, v) in enumerate(zip(df["date"], pos)):
        if not in_trade and v != 0:
            in_trade = True
            t_in = dt
            cur_dir = v
        if in_trade and v == 0:
            segments.append((t_in, df["date"].iloc[i], cur_dir))
            in_trade = False
    if in_trade:
        segments.append((t_in, df["date"].iloc[-1], cur_dir))

    seg_rows = []
    for (tin, tout, d) in segments:
        mask = (df["date"] >= tin) & (df["date"] <= tout)
        pnl = float(df.loc[mask, "strat_ret_log"].sum())
        seg_rows.append({"t_in": tin.date().isoformat(), "t_out_boundary": tout.date().isoformat(), "dir": int(d), "pnl_log": pnl})
    trades_df = pd.DataFrame(seg_rows)

    return {
        "df": df,
        "missing_pred": missing_pred,
        "metrics": {
            "cumulative_log_return": cum_log,
            "cumulative_simple_return": cum_simple,
            "annualized_vol_ddof0": vol_annual,
            "sharpe_like_rf0_ddof0": sharpe_like,
            "max_drawdown": max_dd,
            "exposure_rate": exposure,
            "turnover": turnover,
            "n_roundtrip_trades": int(len(trades_df)),
            "n_position_change_events": n_change_events,
            "n_flips": flips,
        },
        "trades": trades_df,
    }

def report_run(name: str, ohlc: pd.DataFrame, pred_path: Path):
    pred = _load_pred(pred_path)
    d0, d1 = pred["date"].min(), pred["date"].max()
    ohlc_win = ohlc[(ohlc["date"] >= d0) & (ohlc["date"] <= d1)].copy()
    res = _run_backtest(ohlc_win, pred)

    print(f"\n[W9_CHECK] Run: {name}")
    print(f"[W9_CHECK] Window: {d0.date().isoformat()} → {d1.date().isoformat()}")
    print(f"[W9_CHECK] OHLC rows: {len(ohlc_win)} | OHLC min/max: {ohlc_win['date'].min().date()} → {ohlc_win['date'].max().date()}")
    print(f"[W9_CHECK] Prediction rows: {len(pred)}")
    print(f"[W9_CHECK] Missing y_pred after alignment (filled as 0): {res['missing_pred']} / {len(ohlc_win)}")

    df = res["df"]
    print("\n[W9_CHECK] Distribution checks:")
    print("signal_t value counts:")
    print(df["signal_t"].value_counts().sort_index())
    print("position value counts:")
    print(df["position"].value_counts().sort_index())
    print(f"[W9_CHECK] Exposure (position != 0): {res['metrics']['exposure_rate']:.6f}")

    print("\n[W9_CHECK] Core metrics (log-core; cost=0):")
    for k, v in res["metrics"].items():
        print(f"{k}: {v}")

    print("\n[W9_CHECK] Confusion matrix (rows=true, cols=pred) labels [-1,0,1]:")
    print(_confmat(pred))

    print("\n[W9_CHECK] Sanity table HEAD (first 8 rows):")
    print(df[["date","close","ret_log","y_pred","signal_t","position","strat_ret_log","equity"]].head(8).to_string(index=False))

    print("\n[W9_CHECK] Sanity table TAIL (last 8 rows):")
    print(df[["date","close","ret_log","y_pred","signal_t","position","strat_ret_log","equity"]].tail(8).to_string(index=False))

    print("\n[W9_CHECK] Trades head (if any):")
    trades = res["trades"]
    if len(trades) == 0:
        print("NO TRADES")
    else:
        print(trades.head(10).to_string(index=False))

# --- main ---
ROOT = Path.cwd()
if (ROOT / "data").exists() and (ROOT / "results").exists():
    PROJECT_ROOT = ROOT
else:
    PROJECT_ROOT = ROOT.parent

print(f"[W9_CHECK] CWD: {Path.cwd()}")
print(f"[W9_CHECK] PROJECT_ROOT: {PROJECT_ROOT}")

ohlc = _load_ohlc(PROJECT_ROOT)

pred_dir = PROJECT_ROOT / "results" / "predictions"

# E6 V1
report_run("T2_L2_V1 (E6?)", ohlc, pred_dir / "predictions_E6_V1_test.csv")

for f in ["F1","F2","F3","F4"]:
    p = pred_dir / f"predictions_E8_V2_{f}_test.csv"
    if p.exists():
        report_run(f"T2_L2_V2_{f} (E8)", ohlc, p)
    else:
        print(f"\n[W9_CHECK] Missing expected file: {p}")


///////////////////////////////////////////////////////////////////////////////////////////////////////////

In [ ]:
# W9.Task1 — CORE scope-lock (log returns) + fix B1 baseline + write canonical V1 artifacts
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(12):
        if (cur / "config" / "labels_features.yaml").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise FileNotFoundError("Could not locate project ")

ROOT = find_project_root(Path.cwd())
print(f"[W9.Task1] CWD : {Path.cwd()}")
print(f"[W9.Task1] ROOT: {ROOT}")


CORE_RETURN_CONVENTION = "LOG"   
print(f"[W9.Task1] CORE_RETURN_CONVENTION = {CORE_RETURN_CONVENTION} (simple returns must be EXT only)")


PRED_DIR = ROOT / "results" / "predictions"
OHLC_PATH = ROOT / "data" / "processed" / "mru_usd_ohlc_clean.csv"

OUT_TABLES = ROOT / "results" / "tables"
OUT_FIGS   = ROOT / "results" / "figures"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_FIGS.mkdir(parents=True, exist_ok=True)

# Required prediction files for V1 CORE
PRED_T1 = PRED_DIR / "predictions_E1_V1_test.csv"  # T1 = L1_Q LR (E1)
PRED_T2 = PRED_DIR / "predictions_E5_V1_test.csv"  # T2 = L2   LR (E5)

# Canonical output filenames requested
OUT_DAILY_T1 = OUT_TABLES / "backtest_V1_daily_T1_LR.csv"
OUT_DAILY_T2 = OUT_TABLES / "backtest_V1_daily_T2_LR.csv"
OUT_DAILY_B1 = OUT_TABLES / "backtest_V1_daily_B1.csv"
OUT_METRICS  = OUT_TABLES / "backtest_V1_metrics.csv"
OUT_EQUITY_PNG = OUT_FIGS / "equity_V1_core.png"


req = {
    "OHLC_PATH": OHLC_PATH,
    "PRED_T1": PRED_T1,
    "PRED_T2": PRED_T2,
    "PRED_DIR": PRED_DIR,
}
for k, p in req.items():
    print(f"- {k:10s}: exists={p.exists()}  path={p}")

missing = [k for k, p in req.items() if not p.exists()]
if missing:
    print("\nNot provable from provided evidence.")
    raise FileNotFoundError(f"Missing required paths: {missing}")


V1_START = pd.Timestamp("2020-01-01")
V1_END   = pd.Timestamp("2025-11-25")
print(f"\n[W9.Task1] V1 window: {V1_START.date()} → {V1_END.date()}")


ohlc = pd.read_csv(OHLC_PATH, parse_dates=["date"]).sort_values("date").reset_index(drop=True)

required_ohlc_cols = {"date", "open", "high", "low", "close"}
if not required_ohlc_cols.issubset(set(ohlc.columns)):
    print("Not provable from provided evidence.")
    raise ValueError(f"OHLC missing required cols. Have={list(ohlc.columns)} need={sorted(list(required_ohlc_cols))}")

# date integrity
if int(ohlc["date"].isna().sum()) > 0:
    raise RuntimeError("OHLC date parse produced NaT.")
if int(ohlc["date"].duplicated().sum()) > 0:
    raise RuntimeError("OHLC has duplicate dates.")

ohlc["close"] = pd.to_numeric(ohlc["close"], errors="raise").astype(float)
ohlc["ret_log"] = np.log(ohlc["close"]).diff()

ohlc_w = ohlc[(ohlc["date"] >= V1_START) & (ohlc["date"] <= V1_END)].copy().reset_index(drop=True)
if len(ohlc_w) == 0:
    raise RuntimeError("Empty OHLC in V1 window — check dates.")
if int(ohlc_w["ret_log"].isna().sum()) > 0:
    raise RuntimeError("ret_log has NaN inside V1 window. This breaks CORE log-return backtest.")

print(f"[W9.Task1] OHLC window rows: {len(ohlc_w)} | OHLC min/max: {ohlc_w['date'].min().date()} → {ohlc_w['date'].max().date()}")


def load_pred(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
    need = {"date", "y_true", "y_pred"}
    if set(df.columns) != need:
        raise ValueError(f"{path.name} unexpected columns: {list(df.columns)} (expected exactly {sorted(list(need))})")
    if int(df["date"].isna().sum()) > 0:
        raise RuntimeError(f"{path.name} date parse produced NaT")
    if int(df["date"].duplicated().sum()) > 0:
        raise RuntimeError(f"{path.name} has duplicate dates")
    df["y_pred"] = pd.to_numeric(df["y_pred"], errors="coerce")
    if int(df["y_pred"].isna().sum()) > 0:
        raise RuntimeError(f"{path.name} y_pred has NaN after numeric coercion")
    # enforce allowed labels
    vals = set(df["y_pred"].astype(int).unique().tolist())
    allowed = {-1, 0, 1}
    if not vals.issubset(allowed):
        raise RuntimeError(f"{path.name} y_pred values outside {-1,0,1}: {sorted(list(vals))}")
    return df

pred_t1 = load_pred(PRED_T1)
pred_t2 = load_pred(PRED_T2)

print(f"[W9.Task1] PRED_T1 rows: {len(pred_t1)} | min/max: {pred_t1['date'].min().date()} → {pred_t1['date'].max().date()}")
print(f"[W9.Task1] PRED_T2 rows: {len(pred_t2)} | min/max: {pred_t2['date'].min().date()} → {pred_t2['date'].max().date()}")


ANNUALIZATION = 252.0

def compute_roundtrip_trades(df: pd.DataFrame) -> pd.DataFrame:
    pos = df["position"].to_numpy()
    strat = df["strat_ret_log"].to_numpy()

    trades = []
    i = 0
    n = len(df)
    while i < n:
        if pos[i] == 0:
            i += 1
            continue
        direction = pos[i]
        j = i
        while j + 1 < n and pos[j + 1] == direction:
            j += 1
        pnl = float(np.nansum(strat[i:j+1]))
        trades.append({
            "t_in": df.loc[i, "date"],
            "t_out_boundary": (df.loc[j+1, "date"] if j+1 < n else df.loc[j, "date"]),
            "dir": int(direction),
            "pnl_log": pnl,
        })
        i = j + 1
    return pd.DataFrame(trades)

def run_T(pred: pd.DataFrame, name: str) -> tuple[pd.DataFrame, dict]:
    df = ohlc_w.merge(pred[["date", "y_pred"]], on="date", how="left", validate="one_to_one")

    missing_pred = int(df["y_pred"].isna().sum())
    df["y_pred_raw"] = df["y_pred"] 

    df["signal_t"] = df["y_pred"].fillna(0).astype(int)

    df["position"] = df["signal_t"].shift(1).fillna(0).astype(int)

    df["strat_ret_log"] = df["position"].astype(float) * df["ret_log"].astype(float)
    df["equity"] = np.exp(df["strat_ret_log"].fillna(0.0).cumsum())

    metrics = compute_metrics(df, name=name, missing_pred=missing_pred)
    return df, metrics

def run_B1(name: str = "B1") -> tuple[pd.DataFrame, dict]:
    df = ohlc_w.copy()
    df["y_pred_raw"] = 1
    df["signal_t"] = 1
    df["position"] = 1  # FIX: always-long, no exceptions

    df["strat_ret_log"] = df["position"].astype(float) * df["ret_log"].astype(float)
    df["equity"] = np.exp(df["strat_ret_log"].fillna(0.0).cumsum())

    metrics = compute_metrics(df, name=name, missing_pred=0)
    return df, metrics

def compute_metrics(df: pd.DataFrame, name: str, missing_pred: int) -> dict:
    x = df["strat_ret_log"].to_numpy(dtype=float)

    cum_log = float(np.nansum(x))
    cum_simple = float(np.exp(cum_log) - 1.0)

    mu = float(np.nanmean(x))
    sd = float(np.nanstd(x, ddof=0))
    ann_vol = float(sd * np.sqrt(ANNUALIZATION)) if sd > 0 else np.nan
    sharpe = float((mu / sd) * np.sqrt(ANNUALIZATION)) if sd > 0 else np.nan

    eq = df["equity"].astype(float)
    dd = (eq / eq.cummax()) - 1.0
    max_dd = float(dd.min())

    pos = df["position"].to_numpy()
    dpos = np.diff(pos, prepend=pos[0])
    n_change_events = int(np.sum(dpos[1:] != 0))  # exclude first element
    n_flips = int(np.sum(np.abs(dpos[1:]) == 2))  # +1<->-1 changes

    exposure = float(np.mean(pos != 0))
    turnover = float(np.mean(np.abs(dpos)))  # simple, deterministic proxy

    trades = compute_roundtrip_trades(df)
    n_trades = int(len(trades))
    if n_trades == 0:
        win_rate = np.nan
    else:
        win_rate = float((trades["pnl_log"] > 0).mean())

    return {
        "strategy": name,
        "window": "V1_test",
        "return_convention_core": "log",
        "date_min": str(df["date"].min()),
        "date_max": str(df["date"].max()),
        "n_days": int(len(df)),
        "missing_y_pred_filled_neutral": int(missing_pred),
        "cumulative_log_return": cum_log,
        "cumulative_simple_return": cum_simple,
        "annualized_vol_ddof0": ann_vol,
        "sharpe_like_rf0_ddof0": sharpe,
        "max_drawdown": max_dd,
        "exposure_rate": exposure,
        "turnover_proxy_mean_abs_dpos": turnover,
        "n_position_change_events": n_change_events,
        "n_flips": n_flips,
        "n_roundtrip_trades": n_trades,
        "win_rate": win_rate,
    }

# Run V1 CORE
df_t1, m_t1 = run_T(pred_t1, name="T1_L1Q_LR")
df_t2, m_t2 = run_T(pred_t2, name="T2_L2_LR")
df_b1, m_b1 = run_B1(name="B1_USDhold")

# Save daily ledgers (canonical)
keep_cols = ["date","close","ret_log","y_pred_raw","signal_t","position","strat_ret_log","equity"]
df_t1[keep_cols].to_csv(OUT_DAILY_T1, index=False)
df_t2[keep_cols].to_csv(OUT_DAILY_T2, index=False)
df_b1[keep_cols].to_csv(OUT_DAILY_B1, index=False)

# Save metrics (canonical)
metrics = pd.DataFrame([m_t1, m_t2, m_b1])
metrics.to_csv(OUT_METRICS, index=False)

print("\n[W9.Task1] Saved artifacts:")
print(f"- {OUT_DAILY_T1.relative_to(ROOT)}")
print(f"- {OUT_DAILY_T2.relative_to(ROOT)}")
print(f"- {OUT_DAILY_B1.relative_to(ROOT)}")
print(f"- {OUT_METRICS.relative_to(ROOT)}")

# Combined equity plot (canonical)
plt.figure()
plt.plot(df_t1["date"], df_t1["equity"], label="T1_L1Q_LR")
plt.plot(df_t2["date"], df_t2["equity"], label="T2_L2_LR")
plt.plot(df_b1["date"], df_b1["equity"], label="B1_USDhold")
plt.xlabel("date")
plt.ylabel("equity (log-core)")
plt.title("V1 CORE equity curves (log returns, cost=0)")
plt.legend()
plt.tight_layout()
plt.savefig(OUT_EQUITY_PNG, dpi=150)
plt.close()

print(f"- {OUT_EQUITY_PNG.relative_to(ROOT)}")

def endpoints(df: pd.DataFrame, name: str) -> pd.DataFrame:
    out = df[["date","equity","position"]].copy()
    head = out.head(3).assign(which="HEAD3", strategy=name)
    tail = out.tail(3).assign(which="TAIL3", strategy=name)
    return pd.concat([head, tail], axis=0, ignore_index=True)

mirror = pd.concat([
    endpoints(df_t1, "T1_L1Q_LR"),
    endpoints(df_t2, "T2_L2_LR"),
    endpoints(df_b1, "B1_USDhold"),
], axis=0, ignore_index=True)

with pd.option_context("display.width", 180, "display.max_columns", 200):
    print(metrics)

with pd.option_context("display.width", 180):
    print(mirror.to_string(index=False))

# Explicit proof that B1 is always-long 
print("\n[W9.Task1] B1 position value counts (must be only {1}):")
print(df_b1["position"].value_counts(dropna=False).sort_index())



In [ ]:
# W9.Task1-2 — Add B2 (MA crossover) to V1 CORE (log returns), 
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(12):
        if (cur / "config" / "labels_features.yaml").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise FileNotFoundError("Could not locate project root containing ")

ROOT = find_project_root(Path.cwd())

PRED_DIR = ROOT / "results" / "predictions"
OHLC_PATH = ROOT / "data" / "processed" / "mru_usd_ohlc_clean.csv"
L2_LABELS_PATH = ROOT / "data" / "derived" / "mru_usd_labels_L2_ma10_50_h5.csv"

OUT_TABLES = ROOT / "results" / "tables"
OUT_FIGS   = ROOT / "results" / "figures"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_FIGS.mkdir(parents=True, exist_ok=True)

# Canonical output names (V1 core)
OUT_DAILY_T1 = OUT_TABLES / "backtest_V1_daily_T1_LR.csv"
OUT_DAILY_T2 = OUT_TABLES / "backtest_V1_daily_T2_LR.csv"
OUT_DAILY_B1 = OUT_TABLES / "backtest_V1_daily_B1.csv"
OUT_DAILY_B2 = OUT_TABLES / "backtest_V1_daily_B2.csv"          
OUT_METRICS  = OUT_TABLES / "backtest_V1_metrics.csv"         
OUT_EQUITY_PNG = OUT_FIGS / "equity_V1_core.png"               

# Inputs for T1/T2
PRED_T1 = PRED_DIR / "predictions_E1_V1_test.csv"
PRED_T2 = PRED_DIR / "predictions_E5_V1_test.csv"

print(f"[W9.Task1-2] ROOT: {ROOT}")
for name, p in {
    "OHLC": OHLC_PATH,
    "PRED_T1": PRED_T1,
    "PRED_T2": PRED_T2,
    "L2_LABELS": L2_LABELS_PATH,
}.items():
    print(f"- {name:8s} exists={p.exists()}  path={p}")
missing = [n for n,p in {
    "OHLC": OHLC_PATH,
    "PRED_T1": PRED_T1,
    "PRED_T2": PRED_T2,
    "L2_LABELS": L2_LABELS_PATH,
}.items() if not p.exists()]
if missing:
    print("Not provable from provided evidence.")
    raise FileNotFoundError(f"Missing required paths: {missing}")

V1_START = pd.Timestamp("2020-01-01")
V1_END   = pd.Timestamp("2025-11-25")
print(f"\n[W9.Task1-2] V1 window: {V1_START.date()} → {V1_END.date()}")

ohlc = pd.read_csv(OHLC_PATH, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
if int(ohlc["date"].duplicated().sum()) > 0:
    raise RuntimeError("OHLC duplicate dates")
ohlc["close"] = pd.to_numeric(ohlc["close"], errors="raise").astype(float)
ohlc["ret_log"] = np.log(ohlc["close"]).diff()

ohlc_w = ohlc[(ohlc["date"] >= V1_START) & (ohlc["date"] <= V1_END)].copy().reset_index(drop=True)
if len(ohlc_w) == 0:
    raise RuntimeError("Empty OHLC window")
if int(ohlc_w["ret_log"].isna().sum()) > 0:
    raise RuntimeError("ret_log has NaN inside V1 window .")

ANNUALIZATION = 252.0

def compute_roundtrip_trades(df: pd.DataFrame) -> pd.DataFrame:
    pos = df["position"].to_numpy()
    strat = df["strat_ret_log"].to_numpy()

    trades = []
    i = 0
    n = len(df)
    while i < n:
        if pos[i] == 0:
            i += 1
            continue
        direction = pos[i]
        j = i
        while j + 1 < n and pos[j + 1] == direction:
            j += 1
        pnl = float(np.nansum(strat[i:j+1]))
        trades.append({"t_in": df.loc[i, "date"], "t_out_boundary": (df.loc[j+1, "date"] if j+1 < n else df.loc[j, "date"]), "dir": int(direction), "pnl_log": pnl})
        i = j + 1
    return pd.DataFrame(trades)

def compute_metrics(df: pd.DataFrame, name: str, missing_pred: int) -> dict:
    x = df["strat_ret_log"].to_numpy(dtype=float)
    cum_log = float(np.nansum(x))
    cum_simple = float(np.exp(cum_log) - 1.0)

    mu = float(np.nanmean(x))
    sd = float(np.nanstd(x, ddof=0))
    ann_vol = float(sd * np.sqrt(ANNUALIZATION)) if sd > 0 else np.nan
    sharpe = float((mu / sd) * np.sqrt(ANNUALIZATION)) if sd > 0 else np.nan

    eq = df["equity"].astype(float)
    dd = (eq / eq.cummax()) - 1.0
    max_dd = float(dd.min())

    pos = df["position"].to_numpy()
    dpos = np.diff(pos, prepend=pos[0])
    n_change_events = int(np.sum(dpos[1:] != 0))
    n_flips = int(np.sum(np.abs(dpos[1:]) == 2))

    exposure = float(np.mean(pos != 0))
    turnover = float(np.mean(np.abs(dpos)))

    trades = compute_roundtrip_trades(df)
    n_trades = int(len(trades))
    win_rate = float((trades["pnl_log"] > 0).mean()) if n_trades > 0 else np.nan

    return {
        "strategy": name,
        "window": "V1_test",
        "return_convention_core": "log",
        "date_min": str(df["date"].min()),
        "date_max": str(df["date"].max()),
        "n_days": int(len(df)),
        "missing_y_pred_filled_neutral": int(missing_pred),
        "cumulative_log_return": cum_log,
        "cumulative_simple_return": cum_simple,
        "annualized_vol_ddof0": ann_vol,
        "sharpe_like_rf0_ddof0": sharpe,
        "max_drawdown": max_dd,
        "exposure_rate": exposure,
        "turnover_proxy_mean_abs_dpos": turnover,
        "n_position_change_events": n_change_events,
        "n_flips": n_flips,
        "n_roundtrip_trades": n_trades,
        "win_rate": win_rate,
    }

def load_pred(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
    need = {"date","y_true","y_pred"}
    if set(df.columns) != need:
        raise ValueError(f"{path.name} unexpected columns: {list(df.columns)} (expected exactly {sorted(list(need))})")
    if int(df["date"].duplicated().sum()) > 0:
        raise RuntimeError(f"{path.name} duplicate dates")
    df["y_pred"] = pd.to_numeric(df["y_pred"], errors="coerce")
    if int(df["y_pred"].isna().sum()) > 0:
        raise RuntimeError(f"{path.name} y_pred has NaN after numeric coercion")
    vals = set(df["y_pred"].astype(int).unique().tolist())
    if not vals.issubset({-1,0,1}):
        raise RuntimeError(f"{path.name} y_pred outside {-1,0,1}: {sorted(list(vals))}")
    return df

def run_T(pred: pd.DataFrame, name: str) -> tuple[pd.DataFrame, dict]:
    df = ohlc_w.merge(pred[["date","y_pred"]], on="date", how="left", validate="one_to_one")
    missing_pred = int(df["y_pred"].isna().sum())
    df["y_pred_raw"] = df["y_pred"]
    df["signal_t"] = df["y_pred"].fillna(0).astype(int)
    df["position"] = df["signal_t"].shift(1).fillna(0).astype(int)  # 1-day lag
    df["strat_ret_log"] = df["position"].astype(float) * df["ret_log"].astype(float)
    df["equity"] = np.exp(df["strat_ret_log"].fillna(0.0).cumsum())
    return df, compute_metrics(df, name=name, missing_pred=missing_pred)

def run_B1() -> tuple[pd.DataFrame, dict]:
    df = ohlc_w.copy()
    df["y_pred_raw"] = 1
    df["signal_t"] = 1
    df["position"] = 1  
    df["strat_ret_log"] = df["position"].astype(float) * df["ret_log"].astype(float)
    df["equity"] = np.exp(df["strat_ret_log"].fillna(0.0).cumsum())
    return df, compute_metrics(df, name="B1_USDhold", missing_pred=0)

l2 = pd.read_csv(L2_LABELS_PATH, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
if "date" not in l2.columns:
    raise RuntimeError("L2 label file missing 'date' column")

cols = list(l2.columns)
cols_l = [c.lower() for c in cols]

def pick_unique(cands: list[str]) -> str | None:
    uniq = list(dict.fromkeys(cands))
    return uniq[0] if len(uniq) == 1 else None

ma_fast_cands = [cols[i] for i,c in enumerate(cols_l) if c in ("ma_fast","ma10","ma_10","ma10_close","ma_10_close")]
ma_slow_cands = [cols[i] for i,c in enumerate(cols_l) if c in ("ma_slow","ma50","ma_50","ma50_close","ma_50_close")]

if not ma_fast_cands:
    ma_fast_cands = [cols[i] for i,c in enumerate(cols_l) if ("ma10" in c) or ("ma_10" in c) or ("ma_fast" in c)]
if not ma_slow_cands:
    ma_slow_cands = [cols[i] for i,c in enumerate(cols_l) if ("ma50" in c) or ("ma_50" in c) or ("ma_slow" in c)]

ma_fast = pick_unique(ma_fast_cands)
ma_slow = pick_unique(ma_slow_cands)

print("\n[W9.Task1-2] L2 label columns:", cols)
print(f"[W9.Task1-2] MA fast candidates: {ma_fast_cands}")
print(f"[W9.Task1-2] MA slow candidates: {ma_slow_cands}")

if ma_fast is None or ma_slow is None:
    print("Not provable from provided evidence.")
    raise RuntimeError(
        "Could not deterministically select unique MA_fast/MA_slow columns from L2 label file. "

    )

print(f"[W9.Task1-2] Selected MA columns: MA_fast='{ma_fast}'  MA_slow='{ma_slow}'")

def run_B2() -> tuple[pd.DataFrame, dict]:
    # align MA columns onto OHLC window by date
    ma = l2[["date", ma_fast, ma_slow]].copy()
    df = ohlc_w.merge(ma, on="date", how="left", validate="one_to_one")

    # signal = sign(MA_fast - MA_slow); NaN/tie -> 0
    diff = pd.to_numeric(df[ma_fast], errors="coerce") - pd.to_numeric(df[ma_slow], errors="coerce")
    sig = np.sign(diff).replace({-0.0: 0.0})
    sig = sig.where(diff.notna(), 0.0)  # NaN -> 0
    sig = sig.astype(int)

    df["y_pred_raw"] = sig
    df["signal_t"] = sig

    # 1-day lag for trading
    df["position"] = df["signal_t"].shift(1).fillna(0).astype(int)

    df["strat_ret_log"] = df["position"].astype(float) * df["ret_log"].astype(float)
    df["equity"] = np.exp(df["strat_ret_log"].fillna(0.0).cumsum())

    missing_sig = int(diff.isna().sum())  # informative
    metrics = compute_metrics(df, name="B2_MAcross", missing_pred=missing_sig)
    return df, metrics

pred_t1 = load_pred(PRED_T1)
pred_t2 = load_pred(PRED_T2)

df_t1, m_t1 = run_T(pred_t1, name="T1_L1Q_LR")
df_t2, m_t2 = run_T(pred_t2, name="T2_L2_LR")
df_b1, m_b1 = run_B1()
df_b2, m_b2 = run_B2()

keep_cols = ["date","close","ret_log","y_pred_raw","signal_t","position","strat_ret_log","equity"]
df_t1[keep_cols].to_csv(OUT_DAILY_T1, index=False)
df_t2[keep_cols].to_csv(OUT_DAILY_T2, index=False)
df_b1[keep_cols].to_csv(OUT_DAILY_B1, index=False)
df_b2[keep_cols].to_csv(OUT_DAILY_B2, index=False)

metrics = pd.DataFrame([m_t1, m_t2, m_b1, m_b2])
metrics.to_csv(OUT_METRICS, index=False)

print("\n[W9.Task1-2] Saved artifacts:")
print(f"- {OUT_DAILY_T1.relative_to(ROOT)}")
print(f"- {OUT_DAILY_T2.relative_to(ROOT)}")
print(f"- {OUT_DAILY_B1.relative_to(ROOT)}")
print(f"- {OUT_DAILY_B2.relative_to(ROOT)}")
print(f"- {OUT_METRICS.relative_to(ROOT)}")

plt.figure()
plt.plot(df_t1["date"], df_t1["equity"], label="T1_L1Q_LR")
plt.plot(df_t2["date"], df_t2["equity"], label="T2_L2_LR")
plt.plot(df_b1["date"], df_b1["equity"], label="B1_USDhold")
plt.plot(df_b2["date"], df_b2["equity"], label="B2_MAcross")
plt.xlabel("date"); plt.ylabel("equity (log-core)")
plt.title("V1 CORE equity curves (log returns, cost=0)")
plt.legend()
plt.tight_layout()
plt.savefig(OUT_EQUITY_PNG, dpi=150)
plt.close()
print(f"- {OUT_EQUITY_PNG.relative_to(ROOT)}")

with pd.option_context("display.width", 200, "display.max_columns", 200):
    print(metrics)

def endpoints(df: pd.DataFrame, name: str) -> pd.DataFrame:
    out = df[["date","equity","position"]].copy()
    return pd.concat([out.head(3).assign(which="HEAD3", strategy=name),
                      out.tail(3).assign(which="TAIL3", strategy=name)], ignore_index=True)

mirror = pd.concat([
    endpoints(df_t1, "T1_L1Q_LR"),
    endpoints(df_t2, "T2_L2_LR"),
    endpoints(df_b1, "B1_USDhold"),
    endpoints(df_b2, "B2_MAcross"),
], ignore_index=True)

print(mirror.to_string(index=False))

print("signal_t value counts:")
print(df_b2["signal_t"].value_counts(dropna=False).sort_index())
print("position value counts:")
print(df_b2["position"].value_counts(dropna=False).sort_index())



In [ ]:
# W9.Task1-3 — Patch B1 to enforce p_start=0 (first day flat), regenerate canonical V1 artifacts + 
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(12):
        if (cur / "config" / "labels_features.yaml").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise FileNotFoundError("Could not locate project root containing config/labels_features.yaml")

ROOT = find_project_root(Path.cwd())

OUT_TABLES = ROOT / "results" / "tables"
OUT_FIGS   = ROOT / "results" / "figures"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_FIGS.mkdir(parents=True, exist_ok=True)

# Canonical outputs (overwrite)
OUT_DAILY_T1 = OUT_TABLES / "backtest_V1_daily_T1_LR.csv"
OUT_DAILY_T2 = OUT_TABLES / "backtest_V1_daily_T2_LR.csv"
OUT_DAILY_B1 = OUT_TABLES / "backtest_V1_daily_B1.csv"
OUT_DAILY_B2 = OUT_TABLES / "backtest_V1_daily_B2.csv"
OUT_METRICS  = OUT_TABLES / "backtest_V1_metrics.csv"
OUT_EQUITY_PNG = OUT_FIGS / "equity_V1_core.png"

# Load the already-saved daily ledgers (no re-backtest changes except B1 position[0])
t1 = pd.read_csv(OUT_DAILY_T1, parse_dates=["date"])
t2 = pd.read_csv(OUT_DAILY_T2, parse_dates=["date"])
b1 = pd.read_csv(OUT_DAILY_B1, parse_dates=["date"])
b2 = pd.read_csv(OUT_DAILY_B2, parse_dates=["date"])

def recompute_equity(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["strat_ret_log"] = df["position"].astype(float) * df["ret_log"].astype(float)
    df["equity"] = np.exp(df["strat_ret_log"].fillna(0.0).cumsum())
    return df

def compute_metrics(df: pd.DataFrame, name: str, missing_pred: int) -> dict:
    ANNUALIZATION = 252.0
    x = df["strat_ret_log"].to_numpy(dtype=float)
    cum_log = float(np.nansum(x))
    cum_simple = float(np.exp(cum_log) - 1.0)
    mu = float(np.nanmean(x))
    sd = float(np.nanstd(x, ddof=0))
    ann_vol = float(sd * np.sqrt(ANNUALIZATION)) if sd > 0 else np.nan
    sharpe = float((mu / sd) * np.sqrt(ANNUALIZATION)) if sd > 0 else np.nan
    eq = df["equity"].astype(float)
    dd = (eq / eq.cummax()) - 1.0
    max_dd = float(dd.min())
    pos = df["position"].to_numpy()
    dpos = np.diff(pos, prepend=pos[0])
    n_change_events = int(np.sum(dpos[1:] != 0))
    n_flips = int(np.sum(np.abs(dpos[1:]) == 2))
    exposure = float(np.mean(pos != 0))
    turnover = float(np.mean(np.abs(dpos)))

    trades = []
    i, n = 0, len(df)
    while i < n:
        if pos[i] == 0:
            i += 1
            continue
        direction = pos[i]
        j = i
        while j + 1 < n and pos[j + 1] == direction:
            j += 1
        pnl = float(np.nansum(x[i:j+1]))
        trades.append(pnl)
        i = j + 1
    n_trades = int(len(trades))
    win_rate = float(np.mean(np.array(trades) > 0)) if n_trades > 0 else np.nan

    return {
        "strategy": name,
        "window": "V1_test",
        "return_convention_core": "log",
        "date_min": str(df["date"].min()),
        "date_max": str(df["date"].max()),
        "n_days": int(len(df)),
        "missing_y_pred_filled_neutral": int(missing_pred),
        "cumulative_log_return": cum_log,
        "cumulative_simple_return": cum_simple,
        "annualized_vol_ddof0": ann_vol,
        "sharpe_like_rf0_ddof0": sharpe,
        "max_drawdown": max_dd,
        "exposure_rate": exposure,
        "turnover_proxy_mean_abs_dpos": turnover,
        "n_position_change_events": n_change_events,
        "n_flips": n_flips,
        "n_roundtrip_trades": n_trades,
        "win_rate": win_rate,
    }

# ---- PATCH B1 per Frozen Reference §1.2: first day of window must be flat ----
b1_before = b1.copy()
b1["position"] = b1["position"].astype(int)
b1.iloc[0, b1.columns.get_loc("position")] = 0  # enforce p_{t_start}=0
b1 = recompute_equity(b1)

# Save patched B1 daily ledger (canonical name)
b1.to_csv(OUT_DAILY_B1, index=False)

# Recompute metrics (T1/T2/B2 unchanged; B1 changed)
t1 = recompute_equity(t1)
t2 = recompute_equity(t2)
b2 = recompute_equity(b2)

m_t1 = compute_metrics(t1, "T1_L1Q_LR", missing_pred=5)
m_t2 = compute_metrics(t2, "T2_L2_LR", missing_pred=0)
m_b2 = compute_metrics(b2, "B2_MAcross", missing_pred=0)
m_b1 = compute_metrics(b1, "B1_USDhold", missing_pred=0)

metrics = pd.DataFrame([m_t1, m_t2, m_b1, m_b2])
metrics.to_csv(OUT_METRICS, index=False)

# Replot equity (canonical file)
plt.figure()
plt.plot(t1["date"], t1["equity"], label="T1_L1Q_LR")
plt.plot(t2["date"], t2["equity"], label="T2_L2_LR")
plt.plot(b1["date"], b1["equity"], label="B1_USDhold (p_start=0)")
plt.plot(b2["date"], b2["equity"], label="B2_MAcross")
plt.xlabel("date"); plt.ylabel("equity (log-core)")
plt.title("V1 CORE equity curves (log returns, cost=0)")
plt.legend()
plt.tight_layout()
plt.savefig(OUT_EQUITY_PNG, dpi=150)
plt.close()

print("[W9.Task1-3] Patched B1 first day flat: position[0] forced to 0")
print(b1["position"].value_counts(dropna=False).sort_index())

with pd.option_context("display.width", 200, "display.max_columns", 200):
    print(metrics)

def endpoints(df: pd.DataFrame, name: str) -> pd.DataFrame:
    out = df[["date","equity","position"]].copy()
    return pd.concat([out.head(3).assign(which="HEAD3", strategy=name),
                      out.tail(3).assign(which="TAIL3", strategy=name)], ignore_index=True)

mirror = pd.concat([
    endpoints(b1, "B1_USDhold"),
], ignore_index=True)

print(mirror.to_string(index=False))




In [ ]:
# W9.Task1-4 — V2 fold backtests + aggregation (CORE=log), with stable artifacts 
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(12):
        if (cur / "config" / "labels_features.yaml").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise FileNotFoundError("Could not locate project root containing config/labels_features.yaml")

ROOT = find_project_root(Path.cwd())
OHLC_PATH = ROOT / "data" / "processed" / "mru_usd_ohlc_clean.csv"
PRED_DIR  = ROOT / "results" / "predictions"
L2_LABELS = ROOT / "data" / "derived" / "mru_usd_labels_L2_ma10_50_h5.csv"

OUT_TABLES = ROOT / "results" / "tables"
OUT_FIGS   = ROOT / "results" / "figures"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_FIGS.mkdir(parents=True, exist_ok=True)

print(f"[W9.Task1-4] ROOT: {ROOT}")
print(f"[W9.Task1-4] OHLC exists={OHLC_PATH.exists()} | {OHLC_PATH}")
print(f"[W9.Task1-4] PRED_DIR exists={PRED_DIR.exists()} | {PRED_DIR}")
print(f"[W9.Task1-4] L2_LABELS exists={L2_LABELS.exists()} | {L2_LABELS}")

V2_WINDOWS = {
    "F1": (pd.Timestamp("2010-01-01"), pd.Timestamp("2011-12-31")),
    "F2": (pd.Timestamp("2012-01-01"), pd.Timestamp("2013-12-31")),
    "F3": (pd.Timestamp("2014-01-01"), pd.Timestamp("2015-12-31")),
    "F4": (pd.Timestamp("2016-01-01"), pd.Timestamp("2019-12-31")),
}

# Core prediction sources per Week-7 design text:
# T1 (L1_Q, V2): E2
# T2 (L2,   V2): E7
def pred_path(exp: str, fold: str) -> Path:
    return PRED_DIR / f"predictions_{exp}_V2_{fold}_test.csv"

# ---------- Load OHLC + canonical log returns ----------
ohlc = pd.read_csv(OHLC_PATH, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
req_cols = ["date","open","high","low","close"]
if list(ohlc.columns) != req_cols:
    raise ValueError(f"OHLC columns unexpected. Expected={req_cols} got={ohlc.columns.tolist()}")

ohlc["log_close"] = np.log(ohlc["close"].astype(float))
ohlc["ret_log"] = ohlc["log_close"].diff()  # r_t indexed by date t

# ---------- Load L2 MA file for B2 ----------
if not L2_LABELS.exists():
    raise FileNotFoundError(f"Missing L2 labels/MA file for B2: {L2_LABELS}")

l2 = pd.read_csv(L2_LABELS, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
for c in ["date","MA_fast","MA_slow"]:
    if c not in l2.columns:
        raise ValueError(f"L2 labels file missing required column '{c}'. cols={l2.columns.tolist()}")

LABELS = [-1,0,1]

def _load_pred_yhat(p: Path) -> pd.DataFrame:
    if not p.exists():
        raise FileNotFoundError(f"Missing prediction CSV: {p}")
    df = pd.read_csv(p, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
    if "y_pred" not in df.columns:
        raise ValueError(f"{p.name} missing y_pred. cols={df.columns.tolist()}")
    out = df[["date","y_pred"]].copy()
    out["y_pred"] = pd.to_numeric(out["y_pred"], errors="coerce")
    return out

def _slice_ohlc(start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    w = ohlc[(ohlc["date"] >= start) & (ohlc["date"] <= end)].copy().reset_index(drop=True)
    if len(w) == 0:
        raise ValueError(f"Empty OHLC slice for {start.date()}→{end.date()}")
    return w

def _compute_roundtrip_trades(df: pd.DataFrame) -> pd.DataFrame:
    pos = df["position"].to_numpy()
    strat = df["strat_ret_log"].to_numpy()
    rows = []
    i, n = 0, len(df)
    while i < n:
        if pos[i] == 0:
            i += 1
            continue
        d = pos[i]
        j = i
        while j + 1 < n and pos[j + 1] == d:
            j += 1
        pnl = float(np.nansum(strat[i:j+1]))
        rows.append({
            "t_in": df.loc[i, "date"],
            "t_out_boundary": (df.loc[j+1, "date"] if j+1 < n else df.loc[j, "date"]),
            "dir": int(d),
            "pnl_log": pnl,
        })
        i = j + 1
    return pd.DataFrame(rows)

def _metrics_from_daily(df: pd.DataFrame, strategy: str, window: str, missing_pred: int) -> dict:
    x = df["strat_ret_log"].to_numpy(dtype=float)
    cum_log = float(np.nansum(x))
    cum_simple = float(np.exp(cum_log) - 1.0)
    mu = float(np.nanmean(x))
    sd = float(np.nanstd(x, ddof=0))
    ann_vol = float(sd * np.sqrt(252.0)) if sd > 0 else np.nan
    sharpe = float((mu / sd) * np.sqrt(252.0)) if sd > 0 else np.nan

    eq = df["equity"].astype(float)
    dd = (eq / eq.cummax()) - 1.0
    max_dd = float(dd.min())

    pos = df["position"].to_numpy(dtype=float)
    dpos = np.diff(pos, prepend=pos[0])
    n_change_events = int(np.sum(dpos[1:] != 0))
    n_flips = int(np.sum(np.abs(dpos[1:]) == 2))

    exposure = float(np.mean(pos != 0))
    turnover = float(np.mean(np.abs(dpos)))

    trades = _compute_roundtrip_trades(df)
    n_round = int(len(trades))
    win_rate = float(np.mean(trades["pnl_log"] > 0)) if n_round > 0 else np.nan

    return {
        "strategy": strategy,
        "window": window,
        "return_convention_core": "log",
        "date_min": str(df["date"].min()),
        "date_max": str(df["date"].max()),
        "n_days": int(len(df)),
        "missing_y_pred_filled_neutral": int(missing_pred),
        "cumulative_log_return": cum_log,
        "cumulative_simple_return": cum_simple,
        "annualized_vol_ddof0": ann_vol,
        "sharpe_like_rf0_ddof0": sharpe,
        "max_drawdown": max_dd,
        "exposure_rate": exposure,
        "turnover_proxy_mean_abs_dpos": turnover,
        "n_position_change_events": n_change_events,
        "n_flips": n_flips,
        "n_roundtrip_trades": n_round,
        "win_rate": win_rate,
    }

def _save_equity_plot(fold: str, series_list: list[tuple[str,pd.DataFrame]]) -> Path:
    plt.figure()
    for name, df in series_list:
        plt.plot(df["date"], df["equity"], label=name)
    plt.xlabel("date"); plt.ylabel("equity (log-core)")
    plt.title(f"V2 {fold} equity curves (CORE=log, cost=0)")
    plt.legend()
    plt.tight_layout()
    out = OUT_FIGS / f"equity_V2_{fold}_core.png"
    plt.savefig(out, dpi=150)
    plt.close()
    return out

def _endpoints(df: pd.DataFrame, fold: str, strategy: str) -> pd.DataFrame:
    out = df[["date","equity","position"]].copy()
    return pd.concat([
        out.head(3).assign(which="HEAD3", fold=fold, strategy=strategy),
        out.tail(3).assign(which="TAIL3", fold=fold, strategy=strategy),
    ], ignore_index=True)

metrics_rows = []
endpoints_rows = []

missing_files = []
for fold, (s, e) in V2_WINDOWS.items():
    # Require prediction files
    p_t1 = pred_path("E2", fold)   # T1
    p_t2 = pred_path("E7", fold)   # T2

    if not p_t1.exists(): missing_files.append(str(p_t1.relative_to(ROOT)))
    if not p_t2.exists(): missing_files.append(str(p_t2.relative_to(ROOT)))

if missing_files:
    print("Not provable from provided evidence.")
    print("[W9.Task1-4] Missing required V2 prediction files:")
    for x in missing_files:
        print("-", x)
    raise FileNotFoundError("Missing required V2 prediction files (E2 and/or E7).")

for fold, (start, end) in V2_WINDOWS.items():
    win = _slice_ohlc(start, end)

    # T1 (E2): signal_t = y_pred(t), position_t = signal_{t-1}; missing pred -> 0; first day position forced 0 (no signal_{t-1})
    pred_t1 = _load_pred_yhat(pred_path("E2", fold))
    df_t1 = win.merge(pred_t1, on="date", how="left", validate="one_to_one")
    miss_t1 = int(df_t1["y_pred"].isna().sum())
    df_t1["signal_t"] = df_t1["y_pred"].fillna(0).astype(int)
    df_t1["position"] = df_t1["signal_t"].shift(1).fillna(0).astype(int)
    df_t1.loc[0, "position"] = 0
    df_t1["strat_ret_log"] = df_t1["position"].astype(float) * df_t1["ret_log"].astype(float)
    df_t1["equity"] = np.exp(df_t1["strat_ret_log"].fillna(0.0).cumsum())

    # T2 (E7): same mechanics 
    pred_t2 = _load_pred_yhat(pred_path("E7", fold))
    df_t2 = win.merge(pred_t2, on="date", how="left", validate="one_to_one")
    miss_t2 = int(df_t2["y_pred"].isna().sum())
    df_t2["signal_t"] = df_t2["y_pred"].fillna(0).astype(int)
    df_t2["position"] = df_t2["signal_t"].shift(1).fillna(0).astype(int)
    df_t2.loc[0, "position"] = 0
    df_t2["strat_ret_log"] = df_t2["position"].astype(float) * df_t2["ret_log"].astype(float)
    df_t2["equity"] = np.exp(df_t2["strat_ret_log"].fillna(0.0).cumsum())

    # B1: always-long USD in-window.
    # Implement by setting ret_log at the first in-window day to 0 (equity starts at 1), position=+1 all days.
    df_b1 = win[["date","close","ret_log"]].copy()
    df_b1.loc[0, "ret_log"] = 0.0
    df_b1["signal_t"] = 1
    df_b1["position"] = 1
    df_b1["y_pred"] = np.nan
    df_b1["strat_ret_log"] = df_b1["position"].astype(float) * df_b1["ret_log"].astype(float)
    df_b1["equity"] = np.exp(df_b1["strat_ret_log"].fillna(0.0).cumsum())
    miss_b1 = 0

    # B2: MA crossover using MA_fast/MA_slow from L2 file; signal based on MA comparison at t; trade uses lag (t-1)
    ma_win = l2[(l2["date"] >= start) & (l2["date"] <= end)][["date","MA_fast","MA_slow"]].copy()
    df_b2 = win.merge(ma_win, on="date", how="left", validate="one_to_one")
    # signal_t rules
    sig = np.where(df_b2["MA_fast"].isna() | df_b2["MA_slow"].isna(), 0,
          np.where(df_b2["MA_fast"] > df_b2["MA_slow"], 1,
          np.where(df_b2["MA_fast"] < df_b2["MA_slow"], -1, 0)))
    df_b2["signal_t"] = sig.astype(int)
    df_b2["position"] = df_b2["signal_t"].shift(1).fillna(0).astype(int)
    df_b2.loc[0, "position"] = 0
    df_b2["y_pred"] = np.nan
    df_b2["strat_ret_log"] = df_b2["position"].astype(float) * df_b2["ret_log"].astype(float)
    df_b2["equity"] = np.exp(df_b2["strat_ret_log"].fillna(0.0).cumsum())
    miss_b2 = 0

    def save_daily(df: pd.DataFrame, fold: str, name: str) -> Path:
        out = OUT_TABLES / f"backtest_V2_{fold}_daily_{name}.csv"
        keep = ["date","close","ret_log","y_pred","signal_t","position","strat_ret_log","equity"]
        df[keep].to_csv(out, index=False)
        return out

    save_daily(df_t1, fold, "T1_LR")
    save_daily(df_t2, fold, "T2_LR")
    save_daily(df_b1, fold, "B1")
    save_daily(df_b2, fold, "B2")

    metrics_rows.append(_metrics_from_daily(df_t1, "T1_L1Q_LR", f"V2_{fold}_test", miss_t1))
    metrics_rows.append(_metrics_from_daily(df_t2, "T2_L2_LR",  f"V2_{fold}_test", miss_t2))
    metrics_rows.append(_metrics_from_daily(df_b1, "B1_USDhold", f"V2_{fold}_test", miss_b1))
    metrics_rows.append(_metrics_from_daily(df_b2, "B2_MAcross", f"V2_{fold}_test", miss_b2))

    endpoints_rows.append(_endpoints(df_t1, fold, "T1_L1Q_LR"))
    endpoints_rows.append(_endpoints(df_t2, fold, "T2_L2_LR"))
    endpoints_rows.append(_endpoints(df_b1, fold, "B1_USDhold"))
    endpoints_rows.append(_endpoints(df_b2, fold, "B2_MAcross"))

    _save_equity_plot(fold, [
        ("T1_L1Q_LR", df_t1),
        ("T2_L2_LR",  df_t2),
        ("B1_USDhold", df_b1),
        ("B2_MAcross", df_b2),
    ])

metrics_by_fold = pd.DataFrame(metrics_rows).sort_values(["strategy","window"]).reset_index(drop=True)
out_by_fold = OUT_TABLES / "backtest_V2_metrics_by_fold.csv"
metrics_by_fold.to_csv(out_by_fold, index=False)

num_cols = [
    "n_days","missing_y_pred_filled_neutral",
    "cumulative_log_return","cumulative_simple_return",
    "annualized_vol_ddof0","sharpe_like_rf0_ddof0","max_drawdown",
    "exposure_rate","turnover_proxy_mean_abs_dpos",
    "n_position_change_events","n_flips","n_roundtrip_trades","win_rate"
]
agg = (metrics_by_fold
       .groupby("strategy", as_index=False)[num_cols]
       .mean(numeric_only=True))
agg.insert(1, "window", "V2_agg_mean")
agg["return_convention_core"] = "log"
out_agg = OUT_TABLES / "backtest_V2_metrics_agg.csv"
agg.to_csv(out_agg, index=False)

print("\n[W9.Task1-4] Saved artifacts (V2):")
print(f"- {out_by_fold.relative_to(ROOT)}")
print(f"- {out_agg.relative_to(ROOT)}")
for fold in V2_WINDOWS:
    print(f"- results/tables/backtest_V2_{fold}_daily_T1_LR.csv")
    print(f"- results/tables/backtest_V2_{fold}_daily_T2_LR.csv")
    print(f"- results/tables/backtest_V2_{fold}_daily_B1.csv")
    print(f"- results/tables/backtest_V2_{fold}_daily_B2.csv")
    print(f"- results/figures/equity_V2_{fold}_core.png")

with pd.option_context("display.width", 220, "display.max_columns", 200, "display.max_rows", 500):
    print(metrics_by_fold)

with pd.option_context("display.width", 220, "display.max_columns", 200):
    print(agg)

endpoints = pd.concat(endpoints_rows, ignore_index=True)
print(endpoints.to_string(index=False))



In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path(r"C:\Users\simon\Desktop\my thesis\ThesisProject")
TABLES = ROOT / "results" / "tables"

B1_FILES = [
    TABLES / "backtest_V2_F1_daily_B1.csv",
    TABLES / "backtest_V2_F2_daily_B1.csv",
    TABLES / "backtest_V2_F3_daily_B1.csv",
    TABLES / "backtest_V2_F4_daily_B1.csv",
]

def check_b1(path: Path):
    df = pd.read_csv(path, parse_dates=["date"])
    # Basic sanity
    required = ["date", "signal_t", "position", "ret_log", "strat_ret_log", "equity"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        print(f"\n=== {path.name} ===\nMISSING COLS: {missing}\nCols={list(df.columns)}")
        return

    # Invariant: B1 signal_t should be +1 everywhere (buy USD) OR blank; position should be 1 except first day 0
    pos_counts = df["position"].value_counts(dropna=False).sort_index()
    sig_counts = df["signal_t"].value_counts(dropna=False).sort_index()

    # Detect first violation: any day after the first where position != 1
    viol = df.iloc[1:][df.iloc[1:]["position"] != 1]
    first_viol_row = viol.iloc[0] if len(viol) else None

    # Detect if signal_t ever not +1 (excluding NaN)
    sig_non1 = df[df["signal_t"].notna() & (df["signal_t"] != 1)]
    first_sig_non1 = sig_non1.iloc[0] if len(sig_non1) else None

    # Quick check: strat_ret_log should equal ret_log * position (within tolerance)
    # (If position is 1, strat_ret_log should == ret_log; if position is -1, it should be -ret_log)
    calc = df["ret_log"] * df["position"].fillna(0)
    diff = (df["strat_ret_log"] - calc).abs()
    max_diff = float(diff.max())

    print(f"\n=== {path.name} ===")
    print(f"rows={len(df)} date_min={df['date'].min().date()} date_max={df['date'].max().date()}")
    print("\nposition value_counts:\n", pos_counts.to_string())
    print("\nsignal_t value_counts:\n", sig_counts.to_string())
    print(f"\nmax |strat_ret_log - ret_log*position| = {max_diff:.12g}")

    if first_viol_row is None:
        print("\nB1 position invariant OK: position==1 for all days after first-day-flat.")
    else:
        print("\nB1 position invariant FAIL (first day after initial where position!=1):")
        print(first_viol_row[["date","close","ret_log","signal_t","position","strat_ret_log","equity"]].to_string())

    if first_sig_non1 is None:
        print("\nB1 signal_t invariant OK: signal_t is always 1 when not NaN.")
    else:
        print("\nB1 signal_t invariant FAIL (first non-1 signal_t):")
        print(first_sig_non1[["date","close","signal_t","position"]].to_string())

for f in B1_FILES:
    if f.exists():
        check_b1(f)
    else:
        print(f"\nMISSING FILE: {f}")


8888888888888

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path(r"C:\Users\simon\Desktop\my thesis\ThesisProject")
p = ROOT / "results" / "tables" / "backtest_V2_F1_daily_B1.csv"

print("PATH:", p)
st = p.stat()
print("size_bytes:", st.st_size)
print("mtime:", st.st_mtime)

df = pd.read_csv(p)
print("\nHEAD(5):")
print(df.head(5).to_string(index=False))
print("\nAny signal_t != 1 ? ->", (df["signal_t"] != 1).any())
print("Unique signal_t:", sorted(df["signal_t"].dropna().unique().tolist()))
print("Unique position:", sorted(df["position"].dropna().unique().tolist()))


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path(r"C:\Users\simon\Desktop\my thesis\ThesisProject")
TABLES = ROOT / "results" / "tables"

B2_FILES = [
    TABLES / "backtest_V2_F1_daily_B2.csv",
    TABLES / "backtest_V2_F2_daily_B2.csv",
    TABLES / "backtest_V2_F3_daily_B2.csv",
    TABLES / "backtest_V2_F4_daily_B2.csv",
]

def check_b2(path: Path):
    df = pd.read_csv(path, parse_dates=["date"])
    required = ["date","signal_t","position","ret_log","strat_ret_log","equity"]
    miss = [c for c in required if c not in df.columns]
    print(f"\n=== {path.name} ===")
    if miss:
        print("MISSING:", miss)
        print("cols:", list(df.columns))
        return

    print(f"rows={len(df)} date_min={df['date'].min().date()} date_max={df['date'].max().date()}")

    # signal_t should be in {-1,0,1}
    sig_vals = sorted(df["signal_t"].dropna().unique().tolist())
    pos_vals = sorted(df["position"].dropna().unique().tolist())
    print("unique signal_t:", sig_vals)
    print("unique position:", pos_vals)

    print("\nsignal_t value_counts:\n", df["signal_t"].value_counts(dropna=False).sort_index().to_string())
    print("\nposition value_counts:\n", df["position"].value_counts(dropna=False).sort_index().to_string())

    bad_sig = df[df["signal_t"].notna() & ~df["signal_t"].isin([-1,0,1])]
    print("\nbad signal_t rows:", len(bad_sig))

    # strat_ret_log should equal ret_log * position (tolerance)
    calc = df["ret_log"] * df["position"].fillna(0)
    diff = (df["strat_ret_log"] - calc).abs()
    print("max |strat_ret_log - ret_log*position| =", float(diff.max()))

for f in B2_FILES:
    check_b2(f)


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path(r"C:\Users\simon\Desktop\my thesis\ThesisProject")
TABLES = ROOT / "results" / "tables"
L2_PATH = ROOT / "data" / "derived" / "mru_usd_labels_L2_ma10_50_h5.csv"

folds = ["F1","F2","F3","F4"]
b2_files = {f: TABLES / f"backtest_V2_{f}_daily_B2.csv" for f in folds}

l2 = pd.read_csv(L2_PATH, parse_dates=["date"])
assert "y_L2" in l2.columns, f"y_L2 not found; cols={list(l2.columns)}"
l2 = l2[["date","y_L2"]]

for f, p in b2_files.items():
    b2 = pd.read_csv(p, parse_dates=["date"])
    m = b2.merge(l2, on="date", how="left")

    print(f"\n=== {p.name} vs y_L2 ===")
    print("rows:", len(m), "date_min:", m["date"].min().date(), "date_max:", m["date"].max().date())

    print("\ny_L2 value_counts (incl NaN):")
    print(m["y_L2"].value_counts(dropna=False).sort_index().to_string())

    print("\nB2 signal_t value_counts:")
    print(m["signal_t"].value_counts(dropna=False).sort_index().to_string())

    # Agreement rate on rows where y_L2 is not NaN
    valid = m["y_L2"].notna()
    agree = (m.loc[valid, "signal_t"] == m.loc[valid, "y_L2"]).mean()
    print("\nagreement_rate(signal_t == y_L2) on non-NaN y_L2:", float(agree))

    # Pair table to see *how* they differ
    pairs = pd.crosstab(m.loc[valid,"y_L2"], m.loc[valid,"signal_t"])
    print("\n(y_L2 rows) x (signal_t cols) crosstab:")
    print(pairs.to_string())


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(r"C:\Users\simon\Desktop\my thesis\ThesisProject")

OHLC_PATH = ROOT / "data" / "processed" / "mru_usd_ohlc_clean.csv"
L2_PATH   = ROOT / "data" / "derived" / "mru_usd_labels_L2_ma10_50_h5.csv"
TABLES    = ROOT / "results" / "tables"

def load_ohlc():
    df = pd.read_csv(OHLC_PATH, parse_dates=["date"])
    if "close" not in df.columns:
        raise ValueError(f"'close' not in OHLC columns: {df.columns.tolist()}")
    df = df.sort_values("date").reset_index(drop=True)
    # log returns from close
    df["ret_log"] = np.log(df["close"]).diff()
    return df[["date", "close", "ret_log"]]

def load_l2():
    l2 = pd.read_csv(L2_PATH, parse_dates=["date"])
    if "y_L2" not in l2.columns:
        raise ValueError(f"'y_L2' not in L2 file columns: {l2.columns.tolist()}")
    l2 = l2[["date", "y_L2"]].sort_values("date").reset_index(drop=True)
    return l2

def rebuild_b2_for_date_index(date_index: pd.Series, out_path: Path, ohlc: pd.DataFrame, l2: pd.DataFrame):
    dates = pd.to_datetime(date_index).sort_values().reset_index(drop=True)

    df = ohlc.merge(pd.DataFrame({"date": dates}), on="date", how="inner")
    if len(df) != len(dates):
        missing = set(dates) - set(df["date"])
        raise ValueError(f"{out_path.name}: OHLC missing {len(missing)} dates. Example: {sorted(list(missing))[:5]}")

    # Fill first-day ret_log to 0 inside the window 
    df["ret_log"] = df["ret_log"].fillna(0.0)

    df = df.merge(l2, on="date", how="left")
    if df["y_L2"].isna().any():
        n = int(df["y_L2"].isna().sum())
        # For safety: neutral if label unavailable
        df["y_L2"] = df["y_L2"].fillna(0)

    # B2 baseline: exactly L2 rule => allow -1/0/+1
    df["y_pred"] = np.nan
    df["signal_t"] = df["y_L2"].astype(int)

    # Position is next-day execution (shift), first day flat automatically
    df["position"] = df["signal_t"].shift(1).fillna(0).astype(int)

    df["strat_ret_log"] = df["ret_log"] * df["position"]
    df["equity"] = np.exp(df["strat_ret_log"].cumsum())

    out = df[["date","close","ret_log","y_pred","signal_t","position","strat_ret_log","equity"]]
    out.to_csv(out_path, index=False)

    # quick invariants to print
    print(f"\nWROTE: {out_path.name} rows={len(out)} date_min={out['date'].min().date()} date_max={out['date'].max().date()}")
    print("signal_t value_counts:", out["signal_t"].value_counts().sort_index().to_dict())
    print("position value_counts:", out["position"].value_counts().sort_index().to_dict())
    max_diff = (out["strat_ret_log"] - out["ret_log"] * out["position"]).abs().max()
    print("max |strat_ret_log - ret_log*position| =", float(max_diff))

ohlc = load_ohlc()
l2 = load_l2()

# Use existing B1 files as the “date index truth” for each window
# V2 folds
for fold in ["F1","F2","F3","F4"]:
    b1_path = TABLES / f"backtest_V2_{fold}_daily_B1.csv"
    b2_out  = TABLES / f"backtest_V2_{fold}_daily_B2.csv"
    b1 = pd.read_csv(b1_path, parse_dates=["date"])
    rebuild_b2_for_date_index(b1["date"], b2_out, ohlc, l2)

# V1 window
b1_v1_path = TABLES / "backtest_V1_daily_B1.csv"
b2_v1_out  = TABLES / "backtest_V1_daily_B2.csv"
b1v1 = pd.read_csv(b1_v1_path, parse_dates=["date"])
rebuild_b2_for_date_index(b1v1["date"], b2_v1_out, ohlc, l2)


In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\simon\Desktop\my thesis\ThesisProject")
L2 = pd.read_csv(ROOT/"data/derived/mru_usd_labels_L2_ma10_50_h5.csv", parse_dates=["date"])[["date","y_L2"]]

for fold in ["F1","F2","F3","F4"]:
    df = pd.read_csv(ROOT/f"results/tables/backtest_V2_{fold}_daily_B2.csv", parse_dates=["date"])
    m = df.merge(L2, on="date", how="left")
    ok = (m["signal_t"].astype("float") == m["y_L2"]).all()
    print(f"{fold}: signal_t == y_L2 -> {ok}")


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


ROOT = Path("..").resolve()
TABLES = ROOT / "results" / "tables"
FIGS = ROOT / "results" / "figures"

RUN_TAG = "b2fix_20251214"  
OVERWRITE_ORIGINALS = True 

V1_DAILY_FILES = {
    "T1_L1Q_LR": TABLES / "backtest_V1_daily_T1_LR.csv",
    "T2_L2_LR":  TABLES / "backtest_V1_daily_T2_LR.csv",
    "B1_USDhold":TABLES / "backtest_V1_daily_B1.csv",
    "B2_MAcross":TABLES / "backtest_V1_daily_B2.csv",  
}

V2_FOLDS = ["F1", "F2", "F3", "F4"]
V2_DAILY_FILES_BY_FOLD = {
    fold: {
        "T1_L1Q_LR": TABLES / f"backtest_V2_{fold}_daily_T1_LR.csv",
        "T2_L2_LR":  TABLES / f"backtest_V2_{fold}_daily_T2_LR.csv",
        "B1_USDhold":TABLES / f"backtest_V2_{fold}_daily_B1.csv",
        "B2_MAcross":TABLES / f"backtest_V2_{fold}_daily_B2.csv", 
    }
    for fold in V2_FOLDS
}

V1_METRICS_PATH = TABLES / "backtest_V1_metrics.csv"
V2_BYFOLD_PATH  = TABLES / "backtest_V2_metrics_by_fold.csv"
V2_AGG_PATH     = TABLES / "backtest_V2_metrics_agg.csv"


def _require_paths(paths: list[Path]) -> None:
    missing = [p for p in paths if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing required files:\n" + "\n".join(str(p) for p in missing))

def compute_metrics_from_daily_csv(
    daily_csv: Path,
    strategy: str,
    window: str,
    return_convention_core: str = "log",
) -> dict:
    df = pd.read_csv(daily_csv)
    req = ["date", "ret_log", "position", "strat_ret_log", "equity"]
    for c in req:
        if c not in df.columns:
            raise ValueError(f"{daily_csv.name} missing required column: {c}")

    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    r = df["strat_ret_log"].astype(float).to_numpy()
    pos = df["position"].fillna(0).astype(int).to_numpy()
    eq = df["equity"].astype(float).to_numpy()

    n_days = int(len(df))
    date_min = df["date"].min()
    date_max = df["date"].max()

    cum_log = float(np.nansum(r))
    cum_simple = float(np.exp(cum_log) - 1.0)

    daily_std = float(np.nanstd(r, ddof=0))
    ann_vol = float(daily_std * np.sqrt(252)) if daily_std > 0 else np.nan
    sharpe_like = float((np.nanmean(r) / daily_std) * np.sqrt(252)) if daily_std > 0 else np.nan

    # Max drawdown from equity curve
    peak = np.maximum.accumulate(eq)
    dd = (eq / peak) - 1.0
    max_dd = float(np.nanmin(dd)) if len(dd) else np.nan

    # Exposure + turnover proxies
    exposure_rate = float(np.mean(pos != 0))
    dpos = np.diff(pos)
    turnover_mean_abs = float(np.mean(np.abs(dpos))) if len(dpos) else 0.0
    n_change = int(np.sum(dpos != 0))

    # Flips: +1 <-> -1 between consecutive days (ignoring 0 days)
    prev = pos[:-1]
    curr = pos[1:]
    flips = int(np.sum((prev != 0) & (curr != 0) & ((prev * curr) == -1)))

    # A simple "trade count" proxy (kept consistent with change-events style)
    n_roundtrip_trades = n_change

    active = (pos != 0)
    win_rate = float(np.mean(r[active] > 0)) if np.any(active) else np.nan

    return {
        "strategy": strategy,
        "window": window,
        "return_convention_core": return_convention_core,
        "date_min": date_min,
        "date_max": date_max,
        "n_days": n_days,
        "missing_y_pred_filled_neutral": 0,  
        "cumulative_log_return": cum_log,
        "cumulative_simple_return": cum_simple,
        "annualized_vol_ddof0": ann_vol,
        "sharpe_like_rf0_ddof0": sharpe_like,
        "max_drawdown": max_dd,
        "exposure_rate": exposure_rate,
        "turnover_proxy_mean_abs_dpos": turnover_mean_abs,
        "n_position_change_events": n_change,
        "n_flips": flips,
        "n_roundtrip_trades": n_roundtrip_trades,
        "win_rate": win_rate,
    }

def upsert_row(df: pd.DataFrame, row: dict, key_cols=("strategy", "window")) -> pd.DataFrame:
    # Ensure all keys exist as columns
    for k in row.keys():
        if k not in df.columns:
            df[k] = np.nan
    # Insert/update
    mask = np.ones(len(df), dtype=bool)
    for k in key_cols:
        if k not in df.columns:
            raise ValueError(f"Metrics table missing key col: {k}")
        mask &= (df[k].astype(str) == str(row[k]))
    if mask.any():
        idx = np.where(mask)[0][0]
        for k, v in row.items():
            df.loc[idx, k] = v
    else:
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    return df

def save_with_tag(df: pd.DataFrame, base_path: Path, tag: str) -> Path:
    out = base_path.with_name(base_path.stem + f"__{tag}" + base_path.suffix)
    df.to_csv(out, index=False)
    return out

def plot_equity(daily_files: dict[str, Path], title: str, out_path: Path) -> None:
    plt.figure()
    for strat, p in daily_files.items():
        d = pd.read_csv(p)
        d["date"] = pd.to_datetime(d["date"])
        d = d.sort_values("date")
        plt.plot(d["date"], d["equity"], label=strat)
    plt.title(title)
    plt.xlabel("date")
    plt.ylabel("equity")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


_require_paths(
    list(V1_DAILY_FILES.values())
    + [p for fold in V2_FOLDS for p in V2_DAILY_FILES_BY_FOLD[fold].values()]
    + [V1_METRICS_PATH, V2_BYFOLD_PATH, V2_AGG_PATH]
)

v1_metrics = pd.read_csv(V1_METRICS_PATH)
v2_byfold  = pd.read_csv(V2_BYFOLD_PATH)
v2_agg     = pd.read_csv(V2_AGG_PATH)


b2_v1_row = compute_metrics_from_daily_csv(
    daily_csv=V1_DAILY_FILES["B2_MAcross"],
    strategy="B2_MAcross",
    window="V1_test",
)
v1_metrics = upsert_row(v1_metrics, b2_v1_row)

b2_rows = []
for fold in V2_FOLDS:
    row = compute_metrics_from_daily_csv(
        daily_csv=V2_DAILY_FILES_BY_FOLD[fold]["B2_MAcross"],
        strategy="B2_MAcross",
        window=f"V2_{fold}_test",
    )
    b2_rows.append(row)
    v2_byfold = upsert_row(v2_byfold, row)

b2_fold_df = pd.DataFrame(b2_rows)
b2_agg_row = {"strategy": "B2_MAcross", "window": "V2_agg_mean", "return_convention_core": "log"}

for col in v2_agg.columns:
    if col in ["strategy", "window", "return_convention_core"]:
        continue
    if pd.api.types.is_numeric_dtype(v2_agg[col]):
        if col in b2_fold_df.columns and pd.api.types.is_numeric_dtype(b2_fold_df[col]):
            b2_agg_row[col] = float(b2_fold_df[col].mean())
        else:
            b2_agg_row[col] = np.nan

v2_agg = upsert_row(v2_agg, b2_agg_row)

# Write tagged outputs (safe)
out_v1 = save_with_tag(v1_metrics, V1_METRICS_PATH, RUN_TAG)
out_bf = save_with_tag(v2_byfold, V2_BYFOLD_PATH, RUN_TAG)
out_ag = save_with_tag(v2_agg, V2_AGG_PATH, RUN_TAG)

print("WROTE (tagged):")
print(" -", out_v1)
print(" -", out_bf)
print(" -", out_ag)

if OVERWRITE_ORIGINALS:
    v1_metrics.to_csv(V1_METRICS_PATH, index=False)
    v2_byfold.to_csv(V2_BYFOLD_PATH, index=False)
    v2_agg.to_csv(V2_AGG_PATH, index=False)
    print("\nOVERWROTE canonical metrics tables too.")

FIGS.mkdir(parents=True, exist_ok=True)
plot_equity(V1_DAILY_FILES, "Equity (V1 core) — B2 fixed", FIGS / f"equity_V1_core__{RUN_TAG}.png")
print("WROTE:", FIGS / f"equity_V1_core__{RUN_TAG}.png")

for fold in V2_FOLDS:
    plot_equity(
        V2_DAILY_FILES_BY_FOLD[fold],
        f"Equity (V2 {fold}) — B2 fixed",
        FIGS / f"equity_V2_{fold}_core__{RUN_TAG}.png",
    )
    print("WROTE:", FIGS / f"equity_V2_{fold}_core__{RUN_TAG}.png")



In [ ]:
from __future__ import annotations

import os
from pathlib import Path
import numpy as np
import pandas as pd


ROOT = Path(r"C:\Users\simon\Desktop\my thesis\ThesisProject")
TABLES = ROOT / "results" / "tables"



def _read_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
        df = df.sort_values("date").reset_index(drop=True)
    return df

def _max_abs_diff(a: pd.Series, b: pd.Series) -> float:
    x = (a - b).astype(float)
    x = x.replace([np.inf, -np.inf], np.nan)
    return float(np.nanmax(np.abs(x.values))) if np.any(~np.isnan(x.values)) else float("nan")

def _vc(s: pd.Series) -> dict:
    vc = s.value_counts(dropna=False)
    out = {}
    for k, v in vc.items():
        out[str(k)] = int(v)
    return out

def _report_header(title: str):
    print("\n" + "="*90)
    print(title)
    print("="*90)

def audit_strategy_daily(path: Path, *, expected_shift: str = "t_uses_t-1", allow_first_day_override: bool = True) -> dict:
    """
    Audits a backtest daily file with columns:
    date, close, ret_log, y_pred, signal_t, position, strat_ret_log, equity

    """
    df = _read_csv(path)

    req = ["date", "ret_log", "signal_t", "position", "strat_ret_log", "equity"]
    missing = [c for c in req if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {path.name}: {missing}")

    # Basic facts
    n = len(df)
    date_min = df["date"].min()
    date_max = df["date"].max()
    is_monotonic = df["date"].is_monotonic_increasing


    for c in ["signal_t", "position", "ret_log", "strat_ret_log", "equity"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    signal_vc = _vc(df["signal_t"])
    position_vc = _vc(df["position"])

    implied = df["ret_log"] * df["position"]
    max_ret_identity = _max_abs_diff(df["strat_ret_log"], implied)

    # Shift check
    # expected_shift = "t_uses_t-1" means position[t] == signal_t[t-1]
    pos = df["position"].copy()
    sig = df["signal_t"].copy()

    shift_ok_rate = np.nan
    shift_mismatches = None
    first_day_ok = True

    if expected_shift == "t_uses_t-1":
        lhs = pos.iloc[1:].reset_index(drop=True)
        rhs = sig.iloc[:-1].reset_index(drop=True)
        # Compare where both non-NaN
        mask = (~lhs.isna()) & (~rhs.isna())
        if mask.any():
            shift_ok_rate = float((lhs[mask].values == rhs[mask].values).mean())
        else:
            shift_ok_rate = float("nan")

        # First-day override check 
        if allow_first_day_override:
            # We accept position[0] in {-1,0,1} 
            first_day_ok = True

        # Show a small mismatch table (up to 10)
        mism = pd.DataFrame({"date": df["date"].iloc[1:].reset_index(drop=True),
                             "position_t": lhs,
                             "signal_t_minus_1": rhs})
        mism = mism[mask & (mism["position_t"] != mism["signal_t_minus_1"])]
        shift_mismatches = mism.head(10)

    elif expected_shift == "t_uses_t":
        # position[t] == signal_t[t]
        mask = (~pos.isna()) & (~sig.isna())
        if mask.any():
            shift_ok_rate = float((pos[mask].values == sig[mask].values).mean())
        else:
            shift_ok_rate = float("nan")
        mism = pd.DataFrame({"date": df["date"], "position_t": pos, "signal_t": sig})
        mism = mism[mask & (mism["position_t"] != mism["signal_t"])]
        shift_mismatches = mism.head(10)
    else:
        raise ValueError("expected_shift must be one of: {'t_uses_t-1','t_uses_t'}")

    # This holds if equity is compounded with exp(strat_ret_log).
    eq = df["equity"].astype(float)
    eq_prev = eq.shift(1)
    with np.errstate(divide="ignore", invalid="ignore"):
        implied_strat = np.log(eq) - np.log(eq_prev)
    eq_mask = (~implied_strat.isna()) & (~df["strat_ret_log"].isna()) & np.isfinite(implied_strat.values)
    max_equity_identity = _max_abs_diff(implied_strat[eq_mask], df.loc[eq_mask, "strat_ret_log"]) if eq_mask.any() else float("nan")

    out = {
        "file": str(path),
        "rows": int(n),
        "date_min": str(date_min.date()) if pd.notna(date_min) else None,
        "date_max": str(date_max.date()) if pd.notna(date_max) else None,
        "dates_monotonic_increasing": bool(is_monotonic),
        "signal_t_value_counts": signal_vc,
        "position_value_counts": position_vc,
        "max_abs_ret_identity": max_ret_identity,
        "shift_ok_rate": shift_ok_rate,
        "max_abs_equity_identity": max_equity_identity,
        "first_day_position": None if n == 0 else float(pos.iloc[0]) if pd.notna(pos.iloc[0]) else None,
        "first_day_signal_t": None if n == 0 else float(sig.iloc[0]) if pd.notna(sig.iloc[0]) else None,
        "shift_mismatch_head": shift_mismatches,
    }
    return out

def print_audit(out: dict, *, show_mismatch_head: bool = True):
    print(f"FILE: {Path(out['file']).name}")
    print(f"rows={out['rows']} date_min={out['date_min']} date_max={out['date_max']} monotonic={out['dates_monotonic_increasing']}")
    print(f"signal_t value_counts: {out['signal_t_value_counts']}")
    print(f"position value_counts: {out['position_value_counts']}")
    print(f"max |strat_ret_log - ret_log*position| = {out['max_abs_ret_identity']}")
    print(f"shift_ok_rate = {out['shift_ok_rate']}")
    print(f"max |(log(eq)-log(eq_prev)) - strat_ret_log| = {out['max_abs_equity_identity']}")
    print(f"first_day: position={out['first_day_position']} signal_t={out['first_day_signal_t']}")
    if show_mismatch_head and out["shift_mismatch_head"] is not None and len(out["shift_mismatch_head"]) > 0:
        print("\nShift mismatch examples (head):")
        print(out["shift_mismatch_head"].to_string(index=False))


def compare_metrics_files(path_a: Path, path_b: Path, key_cols=("strategy", "window")):
    """
    Compares two metrics CSVs and prints row-level diffs on numeric columns.
    """
    a = _read_csv(path_a)
    b = _read_csv(path_b)

    for k in key_cols:
        if k not in a.columns or k not in b.columns:
            raise ValueError(f"Missing key col '{k}' in one of the files.")

    # Outer merge to detect row mismatches
    m = a.merge(b, on=list(key_cols), how="outer", suffixes=("_A", "_B"), indicator=True)

    print(f"\nCompare metrics: {path_a.name}  vs  {path_b.name}")
    print("Row merge status counts:", m["_merge"].value_counts().to_dict())

    # Compare numeric columns intersection
    num_cols = []
    for c in a.columns:
        if c in key_cols:
            continue
        if c in b.columns:
            # Attempt numeric
            aa = pd.to_numeric(a[c], errors="coerce")
            bb = pd.to_numeric(b[c], errors="coerce")
            if (~aa.isna()).any() or (~bb.isna()).any():
                num_cols.append(c)

    if not num_cols:
        print("No overlapping numeric columns found to diff.")
        return

    # Build diffs
    diffs = []
    for c in num_cols:
        ca = f"{c}_A"
        cb = f"{c}_B"
        if ca not in m.columns or cb not in m.columns:
            continue
        x = pd.to_numeric(m[ca], errors="coerce")
        y = pd.to_numeric(m[cb], errors="coerce")
        d = (x - y).abs()
        maxd = float(np.nanmax(d.values)) if np.any(~np.isnan(d.values)) else 0.0
        diffs.append((c, maxd))

    diffs_sorted = sorted(diffs, key=lambda t: -t[1])
    print("\nTop metric column diffs (abs max):")
    for c, maxd in diffs_sorted[:15]:
        print(f"  {c}: {maxd}")

    top_cols = [c for c, _ in diffs_sorted[:10]]
    bad = None
    for c in top_cols:
        d = (pd.to_numeric(m[f"{c}_A"], errors="coerce") - pd.to_numeric(m[f"{c}_B"], errors="coerce")).abs()
        bad = d if bad is None else (bad.fillna(0) + d.fillna(0))

    if bad is not None:
        idx = bad.fillna(0) > 1e-12
        if idx.any():
            print("\nRows with differences (showing key cols + a few columns):")
            cols_show = list(key_cols)
            for c in top_cols[:5]:
                cols_show += [f"{c}_A", f"{c}_B"]
            print(m.loc[idx, cols_show].head(20).to_string(index=False))
        else:
            print("\nNo row-level numeric diffs detected (within 1e-12) on top columns.")


_report_header("W9 Shift/Alignment Audit for T1/T2 (V1 + V2 folds)")

daily_files = [
    TABLES / "backtest_V1_daily_T1_LR.csv",
    TABLES / "backtest_V1_daily_T2_LR.csv",
    TABLES / "backtest_V2_F1_daily_T1_LR.csv",
    TABLES / "backtest_V2_F1_daily_T2_LR.csv",
    TABLES / "backtest_V2_F2_daily_T1_LR.csv",
    TABLES / "backtest_V2_F2_daily_T2_LR.csv",
    TABLES / "backtest_V2_F3_daily_T1_LR.csv",
    TABLES / "backtest_V2_F3_daily_T1_LR.csv", 
    TABLES / "backtest_V2_F3_daily_T2_LR.csv",
    TABLES / "backtest_V2_F4_daily_T1_LR.csv",
    TABLES / "backtest_V2_F4_daily_T2_LR.csv",
]
daily_files = list(dict.fromkeys(daily_files))

EXPECTED_SHIFT = "t_uses_t-1"

for p in daily_files:
    if not p.exists():
        print(f"\nMISSING: {p}")
        continue
    out = audit_strategy_daily(p, expected_shift=EXPECTED_SHIFT, allow_first_day_override=True)
    print()
    print_audit(out, show_mismatch_head=True)

_report_header("Compare metrics tagged vs untagged (B2 fix impact)")

m1 = TABLES / "backtest_V1_metrics.csv"
m1_fix = TABLES / "backtest_V1_metrics__b2fix_20251214.csv"

m2 = TABLES / "backtest_V2_metrics_by_fold.csv"
m2_fix = TABLES / "backtest_V2_metrics_by_fold__b2fix_20251214.csv"

m3 = TABLES / "backtest_V2_metrics_agg.csv"
m3_fix = TABLES / "backtest_V2_metrics_agg__b2fix_20251214.csv"

for a, b in [(m1, m1_fix), (m2, m2_fix), (m3, m3_fix)]:
    if a.exists() and b.exists():
        compare_metrics_files(a, b, key_cols=("strategy", "window"))
    else:
        print(f"Missing one of: {a.name}, {b.name}")



In [ ]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime

TAG = "b2fix_20251214"
OUT_REL = Path("results") / "tables" / f"week09_release_notes__{TAG}.md"


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / "results").exists() and (p / "data").exists():
            return p
    raise RuntimeError(
        f"Could not locate project root by searching parents of: {start}\n"
        "Expected to find both 'results/' and 'data/' in the root."
    )

ROOT = find_project_root(Path.cwd())
OUT_PATH = (ROOT / OUT_REL).resolve()
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)


canonical_tagged = [
    Path("results") / "tables" / f"backtest_V1_metrics__{TAG}.csv",
    Path("results") / "tables" / f"backtest_V2_metrics_by_fold__{TAG}.csv",
    Path("results") / "tables" / f"backtest_V2_metrics_agg__{TAG}.csv",
    Path("results") / "figures" / f"equity_V1_core__{TAG}.png",
    Path("results") / "figures" / f"equity_V2_F1_core__{TAG}.png",
    Path("results") / "figures" / f"equity_V2_F2_core__{TAG}.png",
    Path("results") / "figures" / f"equity_V2_F3_core__{TAG}.png",
    Path("results") / "figures" / f"equity_V2_F4_core__{TAG}.png",
]

superseded_untagged = [
    Path("results") / "tables" / "backtest_V1_metrics.csv",
    Path("results") / "tables" / "backtest_V2_metrics_by_fold.csv",
    Path("results") / "tables" / "backtest_V2_metrics_agg.csv",
    Path("results") / "figures" / "equity_V1_core.png",
    Path("results") / "figures" / "equity_V2_F1_core.png",
    Path("results") / "figures" / "equity_V2_F2_core.png",
    Path("results") / "figures" / "equity_V2_F3_core.png",
    Path("results") / "figures" / "equity_V2_F4_core.png",
]

daily_v1 = [
    Path("results") / "tables" / "backtest_V1_daily_T1_LR.csv",
    Path("results") / "tables" / "backtest_V1_daily_T2_LR.csv",
    Path("results") / "tables" / "backtest_V1_daily_B1.csv",
    Path("results") / "tables" / "backtest_V1_daily_B2.csv",
]
daily_v2 = []
for fold in ["F1", "F2", "F3", "F4"]:
    daily_v2 += [
        Path("results") / "tables" / f"backtest_V2_{fold}_daily_T1_LR.csv",
        Path("results") / "tables" / f"backtest_V2_{fold}_daily_T2_LR.csv",
        Path("results") / "tables" / f"backtest_V2_{fold}_daily_B1.csv",
        Path("results") / "tables" / f"backtest_V2_{fold}_daily_B2.csv",
    ]

other_week9 = [
    Path("results") / "figures" / "summary_macroF1_E1_E8.png",
    Path("results") / "tables" / "summary_E5_E8_vs_E1_E4.csv",
]
for e in ["E5", "E6"]:
    other_week9 += [
        Path("results") / "figures" / f"confmat_{e}_V1_test.png",
        Path("results") / "tables" / f"confmat_{e}_V1_test.csv",
    ]
for e in ["E7", "E8"]:
    other_week9 += [
        Path("results") / "figures" / f"confmat_{e}_V2_allfolds_test.png",
        Path("results") / "tables" / f"confmat_{e}_V2_allfolds_test.csv",
    ]
    for fold in ["F1", "F2", "F3", "F4"]:
        other_week9 += [
            Path("results") / "figures" / f"confmat_{e}_V2_{fold}_test.png",
            Path("results") / "tables" / f"confmat_{e}_V2_{fold}_test.csv",
        ]
for e in ["E1","E2","E3","E4","E5","E6","E7","E8"]:
    other_week9 += [Path("results") / "tables" / f"metrics_{e}.csv"]
preds = []
for p in [
    "predictions_E1_V1_test.csv",
    "predictions_E2_V1_test.csv",
    "predictions_E3_V1_test.csv",
    "predictions_E4_V1_test.csv",
    "predictions_E5_V1_test.csv",
    "predictions_E6_V1_test.csv",
]:
    preds.append(Path("results") / "predictions" / p)
for e in ["E2","E4","E7","E8"]:
    for fold in ["F1","F2","F3","F4"]:
        preds.append(Path("results") / "predictions" / f"predictions_{e}_V2_{fold}_test.csv")

def existence_lines(paths: list[Path]) -> list[str]:
    lines = []
    for rel in paths:
        p = (ROOT / rel)
        lines.append(f"- `{rel.as_posix()}`  exists={p.exists()}")
    return lines



In [ ]:

from __future__ import annotations

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(25):
        if (cur / "config" / "labels_features.yaml").exists() and (cur / "results").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise FileNotFoundError(
        "Could not locate project root containing config/labels_features.yaml and results/.\n"
        f"Start was: {start.resolve()}"
    )

ROOT = find_project_root(Path.cwd())
PRED_DIR = ROOT / "results" / "predictions"
TABLES_DIR = ROOT / "results" / "tables"
FIG_DIR = ROOT / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("PRED_DIR:", PRED_DIR)
print("TABLES_DIR:", TABLES_DIR)
print("FIG_DIR:", FIG_DIR)


def _require_exists(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

def _pick_col(df: pd.DataFrame, candidates: list[str]) -> str:
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    raise KeyError(f"Required column not found. Needed one of {candidates}. Found columns: {list(df.columns)}")

def _parse_pred_label(fname: str) -> tuple[int, str]:

    m = re.match(r"^predictions_(E(\d+))_(V(\d))(?:_(F(\d)))?_test\.csv$", fname)
    if not m:
        return (10_000, fname.replace(".csv", ""))
    E_full = m.group(1)      # e.g. E7
    E_num = int(m.group(2))  # 7
    V_num = int(m.group(4))  # 1 or 2
    F_num = m.group(6)       # fold number or None
    if V_num == 1:
        label = f"{E_full}_V1"
        sort_key = E_num * 100 + 10
    else:
        fold_i = int(F_num) if F_num is not None else 99
        label = f"{E_full}_V2_F{fold_i}"
        sort_key = E_num * 100 + fold_i
    return (sort_key, label)

def _format_cell(v) -> str:
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "NaN"
    if isinstance(v, (int, np.integer)):
        return str(int(v))
    if isinstance(v, (float, np.floating)):
        return f"{float(v):.6f}"
    return str(v)


pred_files = sorted([p for p in PRED_DIR.glob("predictions_*.csv") if p.is_file()])
if len(pred_files) == 0:
    raise FileNotFoundError(f"No prediction CSVs found under: {PRED_DIR}")

records = []
for p in pred_files:
    df = pd.read_csv(p, usecols=["y_pred"])  
    vals = pd.to_numeric(df["y_pred"], errors="coerce")
    if vals.isna().any():
        raise ValueError(f"{p.name}: y_pred contains non-numeric/NaN values after coercion.")
    bad = set(vals.unique()) - {-1, 0, 1}
    if bad:
        raise ValueError(f"{p.name}: y_pred has values outside {{-1,0,1}}: {sorted(bad)}")
    counts = vals.value_counts().to_dict()
    n = int(len(vals))
    sort_key, label = _parse_pred_label(p.name)
    records.append({
        "sort_key": sort_key,
        "label": label,
        "n": n,
        "pred_-1": int(counts.get(-1, 0)),
        "pred_0": int(counts.get(0, 0)),
        "pred_1": int(counts.get(1, 0)),
    })

dist = pd.DataFrame(records).sort_values("sort_key").reset_index(drop=True)
for k in ["pred_-1", "pred_0", "pred_1"]:
    dist[k + "_pct"] = dist[k] / dist["n"].replace(0, np.nan)

fig1 = plt.figure(figsize=(max(12, 0.6 * len(dist)), 6))
ax = plt.gca()
x = np.arange(len(dist))
bottom = np.zeros(len(dist), dtype=float)

for col, lab in [("pred_-1_pct", "pred=-1"), ("pred_0_pct", "pred=0"), ("pred_1_pct", "pred=+1")]:
    ax.bar(x, dist[col].values, bottom=bottom, label=lab)
    bottom += dist[col].values

ax.set_title("Predicted-class distribution per prediction file (proportions)")
ax.set_ylabel("Proportion of y_pred")
ax.set_ylim(0, 1.0)
ax.set_xticks(x)
ax.set_xticklabels(dist["label"].tolist(), rotation=60, ha="right")
ax.grid(True, axis="y", linestyle=":", linewidth=0.8)
ax.legend(loc="upper right", frameon=True)

for i, n in enumerate(dist["n"].tolist()):
    ax.text(i, 1.01, f"n={n}", ha="center", va="bottom", fontsize=8, rotation=90)

out1 = FIG_DIR / "pred_class_dist_E1_E8.png"
fig1.tight_layout()
fig1.savefig(out1, dpi=300, bbox_inches="tight")
plt.close(fig1)
print("Saved:", out1)



t1_path = TABLES_DIR / "backtest_V1_daily_T1_LR.csv"
t2_path = TABLES_DIR / "backtest_V1_daily_T2_LR.csv"
_require_exists(t1_path)
_require_exists(t2_path)

t1 = pd.read_csv(t1_path)
t2 = pd.read_csv(t2_path)

t1_date = _pick_col(t1, ["date"])
t2_date = _pick_col(t2, ["date"])
t1_pos = _pick_col(t1, ["position"])
t2_pos = _pick_col(t2, ["position"])

t1_close = None
t2_close = None
try:
    t1_close = _pick_col(t1, ["close"])
except KeyError:
    pass
try:
    t2_close = _pick_col(t2, ["close"])
except KeyError:
    pass
if t1_close is None and t2_close is None:
    raise KeyError("Neither daily ledger has a 'close' column; cannot plot price background.")

t1_df = t1[[t1_date, t1_pos] + ([t1_close] if t1_close else [])].copy()
t2_df = t2[[t2_date, t2_pos] + ([t2_close] if t2_close else [])].copy()
t1_df.columns = ["date", "pos_T1"] + (["close_T1"] if t1_close else [])
t2_df.columns = ["date", "pos_T2"] + (["close_T2"] if t2_close else [])

t1_df["date"] = pd.to_datetime(t1_df["date"], errors="raise")
t2_df["date"] = pd.to_datetime(t2_df["date"], errors="raise")

df = pd.merge(t1_df, t2_df, on="date", how="outer").sort_values("date").reset_index(drop=True)

if "close_T2" in df.columns:
    df["close"] = pd.to_numeric(df["close_T2"], errors="raise")
elif "close_T1" in df.columns:
    df["close"] = pd.to_numeric(df["close_T1"], errors="raise")

df["pos_T1"] = pd.to_numeric(df["pos_T1"], errors="raise")
df["pos_T2"] = pd.to_numeric(df["pos_T2"], errors="raise")

for col in ["pos_T1", "pos_T2"]:
    bad = set(df[col].dropna().unique()) - {-1, 0, 1}
    if bad:
        raise ValueError(f"Unexpected position values in {col}: {sorted(bad)}")

fig2 = plt.figure(figsize=(16, 7))
gs = fig2.add_gridspec(2, 1, height_ratios=[2.2, 1.0], hspace=0.08)
ax_price = fig2.add_subplot(gs[0, 0])
ax_pos = fig2.add_subplot(gs[1, 0], sharex=ax_price)

ax_price.plot(df["date"], df["close"])
ax_price.set_title("V1 timeline: Close price + positions (T1_L1Q_LR vs T2_L2_LR)")
ax_price.set_ylabel("Close")
ax_price.grid(True, linestyle=":", linewidth=0.8)

ax_pos.step(df["date"], df["pos_T1"], where="post", label="T1 position")
ax_pos.step(df["date"], df["pos_T2"], where="post", label="T2 position")
ax_pos.set_ylabel("Position")
ax_pos.set_yticks([-1, 0, 1])
ax_pos.grid(True, linestyle=":", linewidth=0.8)
ax_pos.legend(loc="upper right", frameon=True)

plt.setp(ax_price.get_xticklabels(), visible=False)

out2 = FIG_DIR / "position_V1_T1_T2_vs_close.png"
fig2.tight_layout()
fig2.savefig(out2, dpi=300, bbox_inches="tight")
plt.close(fig2)
print("Saved:", out2)



v1m_path = TABLES_DIR / "backtest_V1_metrics__b2fix_20251214.csv"
v2a_path = TABLES_DIR / "backtest_V2_metrics_agg__b2fix_20251214.csv"
_require_exists(v1m_path)
_require_exists(v2a_path)

v1_raw = pd.read_csv(v1m_path, dtype=str)
v2_raw = pd.read_csv(v2a_path, dtype=str)

def build_metric_matrix(raw_df: pd.DataFrame, title: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    if "strategy" not in raw_df.columns:
        raise KeyError(f"{title}: missing required column 'strategy'. Found: {list(raw_df.columns)}")

    wanted = [
        "cumulative_log_return",
        "cumulative_simple_return",
        "max_drawdown",
        "annualized_vol_ddof0",
        "sharpe_like_rf0_ddof0",
        "exposure_rate",
        "turnover_proxy_mean_abs_dpos",
        "win_rate",
        "n_roundtrip_trades",
    ]
    present = [c for c in wanted if c in raw_df.columns]
    if len(present) < 5:
        raise KeyError(
            f"{title}: not enough expected metric columns found.\n"
            f"Expected many of: {wanted}\nFound: {list(raw_df.columns)}"
        )

    preferred_order = ["T1_L1Q_LR", "T2_L2_LR", "B1_USDhold", "B2_MAcross"]
    strat = raw_df["strategy"].tolist()
    ordered = [s for s in preferred_order if s in strat] + [s for s in strat if s not in preferred_order]

    df = raw_df.set_index("strategy").loc[ordered, present].copy()

    num = df.apply(lambda col: pd.to_numeric(col, errors="coerce"))

    return df, num

v1_text, v1_num = build_metric_matrix(v1_raw, "V1 metrics")
v2_text, v2_num = build_metric_matrix(v2_raw, "V2_agg metrics")


higher_better = {
    "cumulative_log_return",
    "cumulative_simple_return",
    "sharpe_like_rf0_ddof0",
    "win_rate",
    "max_drawdown",  
}
lower_better = {
    "annualized_vol_ddof0",
    "turnover_proxy_mean_abs_dpos",
}
descriptive = {
    "exposure_rate",
    "n_roundtrip_trades",
}

def minmax_scale_column(x: pd.Series, mode: str) -> pd.Series:
    vals = x.astype(float)
    m = np.nanmin(vals.values) if np.isfinite(vals.values).any() else np.nan
    M = np.nanmax(vals.values) if np.isfinite(vals.values).any() else np.nan
    if not np.isfinite(m) or not np.isfinite(M) or M == m:
        return pd.Series(np.full(len(vals), 0.5), index=x.index) 
    if mode == "higher":
        return (vals - m) / (M - m)
    if mode == "lower":
        return (M - vals) / (M - m)
    if mode == "value":
        return (vals - m) / (M - m)
    raise ValueError(mode)

def make_scaled(num_df: pd.DataFrame) -> pd.DataFrame:
    scaled = pd.DataFrame(index=num_df.index, columns=num_df.columns, dtype=float)
    for c in num_df.columns:
        if c in higher_better:
            scaled[c] = minmax_scale_column(num_df[c], "higher")
        elif c in lower_better:
            scaled[c] = minmax_scale_column(num_df[c], "lower")
        elif c in descriptive:
            scaled[c] = minmax_scale_column(num_df[c], "value")
        else:
            scaled[c] = minmax_scale_column(num_df[c], "value")
    return scaled

v1_scaled = make_scaled(v1_num)
v2_scaled = make_scaled(v2_num)

def plot_heatmap(ax, scaled: pd.DataFrame, text: pd.DataFrame, panel_title: str):
    data = scaled.values.astype(float)
    im = ax.imshow(data, aspect="auto", interpolation="nearest")  
    ax.set_title(panel_title)

    ax.set_yticks(np.arange(scaled.shape[0]))
    ax.set_yticklabels(scaled.index.tolist())
    ax.set_xticks(np.arange(scaled.shape[1]))
    ax.set_xticklabels(scaled.columns.tolist(), rotation=45, ha="right")

    ax.set_xticks(np.arange(-0.5, scaled.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, scaled.shape[0], 1), minor=True)
    ax.grid(which="minor", linestyle=":", linewidth=0.6)
    ax.tick_params(which="minor", bottom=False, left=False)

    for i in range(scaled.shape[0]):
        for j in range(scaled.shape[1]):
            raw_str = text.iloc[i, j]
            raw_num = pd.to_numeric(pd.Series([raw_str]), errors="coerce").iloc[0]
            ax.text(j, i, _format_cell(raw_num) if np.isfinite(raw_num) else str(raw_str),
                    ha="center", va="center", fontsize=7)

    return im

fig3 = plt.figure(figsize=(18, 10))
gs = fig3.add_gridspec(2, 1, height_ratios=[1, 1], hspace=0.25)

ax1 = fig3.add_subplot(gs[0, 0])
im1 = plot_heatmap(ax1, v1_scaled, v1_text, "V1 backtest metrics (color = per-metric min-max scaling; annotated = raw values)")

ax2 = fig3.add_subplot(gs[1, 0])
im2 = plot_heatmap(ax2, v2_scaled, v2_text, "V2 aggregate metrics (color = per-metric min-max scaling; annotated = raw values)")

cbar = fig3.colorbar(im2, ax=[ax1, ax2], fraction=0.02, pad=0.02)
cbar.set_label("Scaled score (0..1)")

out3 = FIG_DIR / "summary_backtest_metrics_dashboard.png"
fig3.tight_layout()
fig3.savefig(out3, dpi=300, bbox_inches="tight")
plt.close(fig3)
print("Saved:", out3)

print("  1)", out1)
print("  2)", out2)
print("  3)", out3)
